# ETL — S&P 500 (mensual 2015–2025) con Refinitiv

**Objetivo:** construir un panel mensual (firm × month) a partir de datos 100% Refinitiv (Eikon/RDP) y exportar el dataset final para estimaciones econométricas.

**Inputs principales:**
- Universo de firmas: `empresas_tickers_test.csv`
- (Opcional) Excels de bonos corporativos: carpeta local (ver Config)

**Outputs principales:**
- `data/clean/panel_master.parquet`
- `data/clean/data_dictionary.csv`
- `outputs/qa/qa_report.csv` (o similar)

**Notas:**
- Requiere credenciales Refinitiv vía variable de entorno `EIKON_APP_KEY`.
- Algunas descargas pueden tardar y están sujetas a rate limits (HTTP 429). El notebook aplica pausas/reintentos donde corresponde.

## 2) Imports

In [2]:
# ======================
# IMPORTS + REFINTIV INIT
# ======================

from pathlib import Path
import os
from dotenv import load_dotenv, find_dotenv
import eikon as ek

# cargar .env
env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)

EIKON_APP_KEY = os.environ.get("EIKON_APP_KEY")

print("Using key prefix:", EIKON_APP_KEY[:8])

# inicializar Refinitiv
ek.set_app_key(EIKON_APP_KEY)

Using key prefix: 00f40e04


In [3]:
# --- Standard library
from __future__ import annotations

import os
import time
import logging
from pathlib import Path
from typing import Iterable, Optional

# --- Third-party
import numpy as np
import pandas as pd

# --- Refinitiv (Eikon)
import eikon as ek

## 3) Config

In [4]:
# ======================
# CONFIG
# ======================

# ---- Horizonte temporal del panel (mensual 2015–2025)
START_DATE = "2015-01-01"
END_DATE   = "2025-12-31"

# -------------------------------------------------
# Root del proyecto
# (si el notebook está en /notebooks subimos un nivel)
# -------------------------------------------------
PROJECT_ROOT = Path.cwd().resolve().parent

# ---- Estructura de datos dentro del repo
DATA_DIR   = PROJECT_ROOT / "data"
INPUT_DIR  = DATA_DIR / "inputs"
RAW_DIR    = DATA_DIR / "raw"
CLEAN_DIR  = DATA_DIR / "clean"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
QA_DIR     = OUTPUT_DIR / "qa"
LOG_DIR    = OUTPUT_DIR / "logs"

# crear carpetas si no existen
for p in (INPUT_DIR, RAW_DIR, CLEAN_DIR, QA_DIR, LOG_DIR):
    p.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------
# INPUTS
# -------------------------------------------------

# universo inicial de empresas
UNIVERSE_FILE = INPUT_DIR / "empresas_tickers_test.csv"

# exports de Workspace con bonos por sector
BONDS_DIR = INPUT_DIR / "bonds_empresas"


# -------------------------------------------------
# OUTPUTS
# -------------------------------------------------

OUT = {

    # universe / metadata
    "universe_firms": CLEAN_DIR / "universe_firms.parquet",
    "firms_metadata": CLEAN_DIR / "firms_metadata.parquet",
    "bonds_metadata": CLEAN_DIR / "bonds_metadata.parquet",

    # equity
    "equity_prices_daily": RAW_DIR / "equity_prices_daily.parquet",
    "equity_returns_monthly": CLEAN_DIR / "equity_returns_monthly.parquet",

    # market
    "market_prices_daily": RAW_DIR / "market_prices_daily.parquet",
    "market_returns_monthly": CLEAN_DIR / "market_returns_monthly.parquet",

    # bonos / spreads alternativos
    "bonds_universe_full": CLEAN_DIR / "bonds_universe_full.parquet",
    "oas_spreads_monthly_bond": RAW_DIR / "oas_spreads_monthly_bond.parquet",
    "cds_spreads_daily": RAW_DIR / "cds_spreads_daily.parquet",

    # master panel
    "panel_master": CLEAN_DIR / "panel_master.parquet",

    # metadata
    "data_dictionary": CLEAN_DIR / "data_dictionary.csv",

    # QA
    "qa_report": QA_DIR / "qa_report.csv",
}

# -------------------------------------------------
# Credenciales Refinitiv
# -------------------------------------------------

EIKON_APP_KEY = os.environ.get("EIKON_APP_KEY")

assert EIKON_APP_KEY, (
    "Falta la variable de entorno EIKON_APP_KEY. "
    "Ejemplo (Windows PowerShell): $env:EIKON_APP_KEY='TU_KEY'"
)

# -------------------------------------------------
# Parámetros operativos
# -------------------------------------------------

REQUEST_PAUSE_SEC = 0.25
RETRY_MAX = 5
RETRY_BACKOFF_SEC = 2.0


# -------------------------------------------------
# Verificación rápida de rutas (debug útil)
# -------------------------------------------------

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("INPUT_DIR:", INPUT_DIR)
print("BONDS_DIR:", BONDS_DIR)
print("Existe BONDS_DIR?", BONDS_DIR.exists())

PROJECT_ROOT: C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main
DATA_DIR: C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\data
INPUT_DIR: C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\data\inputs
BONDS_DIR: C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\data\inputs\bonds_empresas
Existe BONDS_DIR? True


## 4) Logging y algunos helpers

In [5]:
# ======================
# LOGGING + HELPERS
# ======================

def _build_logger(name: str = "etl") -> logging.Logger:
    logger = logging.getLogger(name)
    if logger.handlers:
        return logger  # evita duplicar handlers si re-ejecutás la celda

    logger.setLevel(logging.INFO)

    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s", "%Y-%m-%d %H:%M:%S")

    sh = logging.StreamHandler()
    sh.setFormatter(fmt)
    logger.addHandler(sh)

    fh = logging.FileHandler(LOG_DIR / "etl.log", encoding="utf-8")
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    return logger


logger = _build_logger("etl")


def ensure_datetime(df: pd.DataFrame, col: str) -> pd.DataFrame:
    """
    Fuerza `df[col]` a datetime (naive, sin tz). No altera otras columnas.
    """
    if col not in df.columns:
        raise KeyError(f"Columna '{col}' no existe en df.")
    out = df.copy()
    out[col] = pd.to_datetime(out[col], errors="coerce")
    # Si viniera con tz, lo baja a naive
    if getattr(out[col].dt, "tz", None) is not None:
        out[col] = out[col].dt.tz_localize(None)
    return out


def safe_write_parquet(df: pd.DataFrame, path: Path, index: bool = False) -> None:
    """
    Escribe parquet y loguea tamaño y destino.
    """
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=index)
    logger.info(f"Output -> {path.as_posix()} | rows={len(df):,} cols={df.shape[1]}")


def sleep_request(pause_sec: float = REQUEST_PAUSE_SEC) -> None:
    time.sleep(pause_sec)

## 5) Conexión a refinitiv

In [6]:
# ======================
# REFINTIV CONNECTION
# ======================

ek.set_app_key(EIKON_APP_KEY)
logger.info("Refinitiv/Eikon: APP_KEY seteada.")


2026-03-19 18:35:30 | INFO | Refinitiv/Eikon: APP_KEY seteada.
2026-03-19 18:35:30,571 P[23992] [MainThread 21972] Refinitiv/Eikon: APP_KEY seteada.


## 6) Bonos corporativos — consolidación de emisores y tickers (Workspace export)

**Input:** Excels exportados manualmente desde Refinitiv Workspace (LSEG) por sector, ubicados en `data/inputs/bonds_empresas/`.  
**Output:** `data/inputs/empresas_tickers_test.csv` (Issuer, Ticker), usado como universo de firmas.  
**Chequeos:** columnas requeridas presentes; filas totales por archivo; duplicados y valores faltantes.  
**Notas:** esta etapa no consulta la API. Se usa export manual por falta de permisos para bajar metadata completa vía Python.

In [7]:
# ======================
# BONOS (Workspace export) -> Universo de bonos con metadata
# ======================

import pandas as pd
from pathlib import Path

BONDS_DIR = RAW_DIR / "bonds_empresas"

BOND_FILES = {
    "ConsumerStaples_and_Discretionary.xlsx": "CS_CD",
    "Energy.xlsx": "Energy",
    "Financials.xlsx": "Financials",
    "Health_Care.xlsx": "HealthCare",
    "Industrials.xlsx": "Industrials",
    "Technology_and_comms.xlsx": "Tech_Comms",
    "Utilities_and_Real_Estate.xlsx": "Utilities_RealEstate",
}

REQUIRED_COLS = ["Issuer", "Ticker"]
BOND_RIC_CANDIDATES = ["Instrument", "RIC", "Preferred RIC", "Issue RIC"]
MATURITY_CANDIDATES = ["Maturity", "Maturity Date"]
ISIN_CANDIDATES = ["ISIN"]
COUPON_CANDIDATES = ["Coupon"]
ISSUE_DATE_CANDIDATES = ["Issue Date"]

def first_present(cands, cols):
    for c in cands:
        if c in cols:
            return c
    return None

def read_bond_export(path: Path, sector: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"No existe el archivo: {path}")

    df = pd.read_excel(path)
    df.columns = [str(c).strip() for c in df.columns]

    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(
            f"Faltan columnas {missing} en {path.name}. "
            f"Columnas disponibles: {list(df.columns)}"
        )

    bond_ric_col   = first_present(BOND_RIC_CANDIDATES, list(df.columns))
    maturity_col   = first_present(MATURITY_CANDIDATES, list(df.columns))
    isin_col       = first_present(ISIN_CANDIDATES, list(df.columns))
    coupon_col     = first_present(COUPON_CANDIDATES, list(df.columns))
    issue_date_col = first_present(ISSUE_DATE_CANDIDATES, list(df.columns))

    keep_cols = REQUIRED_COLS.copy()
    for c in [bond_ric_col, maturity_col, isin_col, coupon_col, issue_date_col]:
        if c is not None and c not in keep_cols:
            keep_cols.append(c)

    out = df[keep_cols].copy()
    out["sector_source"] = sector

    rename_map = {}
    if bond_ric_col is not None:
        rename_map[bond_ric_col] = "bond_ric"
    if maturity_col is not None:
        rename_map[maturity_col] = "Maturity"
    if isin_col is not None:
        rename_map[isin_col] = "ISIN"
    if coupon_col is not None:
        rename_map[coupon_col] = "Coupon"
    if issue_date_col is not None:
        rename_map[issue_date_col] = "Issue_Date"

    out = out.rename(columns=rename_map)

    out["Issuer"] = out["Issuer"].astype(str).str.strip()
    out["Ticker"] = out["Ticker"].astype(str).str.strip()

    if "bond_ric" in out.columns:
        out["bond_ric"] = (
            out["bond_ric"]
            .astype(str)
            .str.strip()
            .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
        )

    if "Maturity" in out.columns:
        out["Maturity"] = pd.to_datetime(out["Maturity"], errors="coerce")

    if "Issue_Date" in out.columns:
        out["Issue_Date"] = pd.to_datetime(out["Issue_Date"], errors="coerce")

    if "Coupon" in out.columns:
        out["Coupon"] = pd.to_numeric(out["Coupon"], errors="coerce")

    return out

frames = []
for fname, sector in BOND_FILES.items():
    fp = BONDS_DIR / fname
    tmp = read_bond_export(fp, sector)
    tmp["source_file"] = fname
    frames.append(tmp)

df_bonds_universe = pd.concat(frames, ignore_index=True)

# limpieza básica
if "bond_ric" in df_bonds_universe.columns:
    df_bonds_universe = df_bonds_universe.dropna(subset=["bond_ric"])

df_bonds_universe = df_bonds_universe.drop_duplicates(
    subset=["Issuer", "Ticker", "bond_ric"]
).reset_index(drop=True)

print("\n=== QA df_bonds_universe ===")
print("Shape:", df_bonds_universe.shape)
print("Columnas:", df_bonds_universe.columns.tolist())
print("RICs únicos:", df_bonds_universe["bond_ric"].nunique() if "bond_ric" in df_bonds_universe.columns else "sin bond_ric")
print("NA Maturity:", round(df_bonds_universe["Maturity"].isna().mean(), 4) if "Maturity" in df_bonds_universe.columns else "sin Maturity")
show_cols = [c for c in ["Issuer", "Ticker", "bond_ric", "Maturity", "ISIN", "Coupon", "Issue_Date", "sector_source"] if c in df_bonds_universe.columns]
print(df_bonds_universe[show_cols].head())


=== QA df_bonds_universe ===
Shape: (276, 9)
Columnas: ['Issuer', 'Ticker', 'bond_ric', 'Maturity', 'ISIN', 'Coupon', 'Issue_Date', 'sector_source', 'source_file']
RICs únicos: 276
NA Maturity: 0.0
                Issuer Ticker    bond_ric            Maturity          ISIN  \
0      McDonald's Corp    MCD  580135BY6= 2028-01-07 23:59:12  US580135BY65   
1  Procter & Gamble Co     PG  742718FZ7= 2028-01-25 23:59:12  US742718FZ79   
2          PepsiCo Inc    PEP  713448GA0= 2028-02-06 23:59:12  US713448GA00   
3          PepsiCo Inc    PEP  713448FL7= 2028-02-17 23:59:12  US713448FL73   
4         Coca-Cola Co     KO  191216DJ6= 2028-03-04 23:59:12  US191216DJ60   

   Coupon          Issue_Date sector_source  
0   6.375 1998-01-07 23:59:12         CS_CD  
1   3.950 2023-01-25 23:59:12         CS_CD  
2   4.450 2025-02-06 23:59:12         CS_CD  
3   3.600 2022-07-17 23:59:12         CS_CD  
4   1.500 2021-03-04 23:59:12         CS_CD  


## 7) Empresas — descarga de metadata (Refinitiv)

**Input:** `data/inputs/empresas_tickers_test.csv` (Issuer, Ticker).  
**Output:** `data/raw/company_metadata.parquet` (+ `.csv`).  
**Chequeos:** cobertura de columnas clave (nombre, ISIN, CUSIP), listado de tickers con metadata incompleta y reintentos con sufijos.  
**Notas:** la descarga se hace por batches para evitar rate limits; algunos tickers requieren sufijos de mercado (ej. `.O`, `.N`, `.K`).

In [8]:
# ======================
# EMPRESAS -> metadata Refinitiv
# ======================

FIELDS = [
    "TR.CommonName",
    "TR.ISIN",
    "TR.CUSIP",
    "TR.PermID",
    "TR.HeadquartersCountry",
    "TR.GICSSectorName",
    "TR.GICSIndustryGroupName",
    "TR.GICSIndustryName",
    "TR.GICSSubIndustryName"
]

REQUIRED_META_COLS = ["ISIN", "CUSIP"]

# -----------------------------------
# Helpers
# -----------------------------------

def ticker_to_ric(ticker: str):

    ticker = ticker.strip().upper()

    nasdaq = {"AAPL","MSFT","GOOGL","META","CSCO","PEP","HON"}
    nyse   = {"JPM","BAC","C","WFC","GS","MS","XOM","CVX","COP","OXY",
              "JNJ","PFE","MRK","ABBV","PG","KO","MCD","WMT",
              "GE","MMM","CAT","DUK","NEE","SO","ORCL"}

    if ticker in nasdaq:
        return f"{ticker}.O"

    if ticker in nyse:
        return f"{ticker}.N"

    return f"{ticker}.N"


def normalize_str(s):

    out = s.astype("string").str.strip()
    out = out.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})

    return out


def fetch_metadata_retry(rics, fields, retries=3):

    for attempt in range(retries):

        try:

            df, err = ek.get_data(rics, fields)

            if err is not None and len(err):
                logger.warning(f"Refinitiv error: {err}")

            if df is not None and not df.empty:
                return df

            else:
                logger.warning(f"Intento {attempt+1}: dataframe vacío")

        except Exception as e:

            logger.warning(f"Intento {attempt+1}/{retries} falló con excepción: {e}")

        sleep_request(2)

    raise RuntimeError("Refinitiv no respondió después de varios intentos")


# -----------------------------------
# Helper: dividir listas en batches
# -----------------------------------

def chunked(iterable, size):

    for i in range(0, len(iterable), size):
        yield iterable[i:i + size]


# -----------------------------------
# 1) Leer universo
# -----------------------------------

df_universe = pd.read_csv(UNIVERSE_FILE, dtype=str)

missing_cols = [c for c in ["Ticker","Issuer"] if c not in df_universe.columns]

if missing_cols:
    raise ValueError(f"Faltan columnas {missing_cols}")

df_universe = df_universe.dropna(subset=["Ticker","Issuer"]).reset_index(drop=True)

rics = (
    df_universe["Ticker"]
    .astype(str)
    .str.strip()
    .apply(ticker_to_ric)
    .unique()
    .tolist()
)

logger.info(f"Universo cargado: {len(rics)} tickers")
logger.info(f"Primeros RICs enviados: {rics[:20]}")
logger.info(f"Total RICs enviados: {len(rics)}")


# -----------------------------------
# 2) Descarga metadata
# -----------------------------------

meta_frames = []

for i, batch in enumerate(chunked(rics, 10), start=1):

    logger.info(f"Batch {i} | instrumentos={len(batch)}")

    df = fetch_metadata_retry(batch, FIELDS)

    if df is not None and not df.empty:
        meta_frames.append(df)

    sleep_request(0.5)


if not meta_frames:
    raise RuntimeError("No se descargó metadata de ninguna empresa.")

meta = pd.concat(meta_frames, ignore_index=True)

logger.info(f"Metadata descargada: filas={len(meta)}")


# -----------------------------------
# 3) Normalización
# -----------------------------------

meta.columns = [c.replace("TR.", "").strip() for c in meta.columns]

if "Instrument" in meta.columns:
    meta = meta.rename(columns={"Instrument": "ric"})
elif "RIC" in meta.columns:
    meta = meta.rename(columns={"RIC": "ric"})

if "ric" in meta.columns:
    meta["ric"] = normalize_str(meta["ric"])

for col in ["ISIN", "CUSIP"]:
    if col in meta.columns:
        meta[col] = normalize_str(meta[col])


# -----------------------------------
# 4) QA rápido
# -----------------------------------

bad_mask = pd.Series(False, index=meta.index)

for col in REQUIRED_META_COLS:

    if col not in meta.columns:
        logger.warning(f"Falta columna esperada: {col}")
        continue

    bad_mask |= meta[col].isna()

bad_rows = meta[bad_mask]

logger.info(f"Empresas con metadata incompleta: {len(bad_rows)}")


# -----------------------------------
# 5) Export
# -----------------------------------

out_parquet = RAW_DIR / "company_metadata.parquet"
out_csv     = RAW_DIR / "company_metadata.csv"

safe_write_parquet(meta, out_parquet, index=False)
meta.to_csv(out_csv, index=False)

logger.info(f"Output -> {out_csv}")

2026-03-19 18:35:35 | INFO | Universo cargado: 32 tickers
2026-03-19 18:35:35,400 P[23992] [MainThread 21972] Universo cargado: 32 tickers
2026-03-19 18:35:35 | INFO | Primeros RICs enviados: ['MCD.N', 'PG.N', 'PEP.O', 'KO.N', 'WMT.N', 'OXY.N', 'COP.N', 'XOM.N', 'CVX.N', 'BAC.N', 'C.N', 'GS.N', 'JPM.N', 'MS.N', 'WFC.N', 'JNJ.N', 'MRK.N', 'ABBV.N', 'PFE.N', 'HON.O']
2026-03-19 18:35:35,402 P[23992] [MainThread 21972] Primeros RICs enviados: ['MCD.N', 'PG.N', 'PEP.O', 'KO.N', 'WMT.N', 'OXY.N', 'COP.N', 'XOM.N', 'CVX.N', 'BAC.N', 'C.N', 'GS.N', 'JPM.N', 'MS.N', 'WFC.N', 'JNJ.N', 'MRK.N', 'ABBV.N', 'PFE.N', 'HON.O']
2026-03-19 18:35:35 | INFO | Total RICs enviados: 32
2026-03-19 18:35:35,403 P[23992] [MainThread 21972] Total RICs enviados: 32
2026-03-19 18:35:35 | INFO | Batch 1 | instrumentos=10
2026-03-19 18:35:35,404 P[23992] [MainThread 21972] Batch 1 | instrumentos=10
2026-03-19 18:35:41 | INFO | Batch 2 | instrumentos=10
2026-03-19 18:35:41,466 P[23992] [MainThread 21972] Batch 2 | i

## 8) Equity — precios diarios (CLOSE, VOLUME)

**Input:** `data/inputs/empresas_tickers_test.csv` (Ticker).  
**Output:** `data/raw/equity_prices_daily.parquet` (+ `.csv`).  
**Chequeos:** conteo de RICs con datos vs sin datos; rango de fechas por RIC; listado de series que arrancan tarde.  
**Notas:** para algunos tickers se prueban sufijos `.N` y `.O`. Se aplica pausa entre requests para reducir riesgos de rate limit (HTTP 429).

In [17]:
# ======================
# EQUITY -> precios diarios (CLOSE, VOLUME)
# ======================

# Universo de tickers (sale del CSV de inputs)
df_emp = pd.read_csv(UNIVERSE_FILE, dtype=str).dropna(subset=["Ticker"]).copy()
df_emp["Ticker"] = df_emp["Ticker"].astype(str).str.strip()

tickers_raw = df_emp["Ticker"].tolist()

logger.info(f"Empresas en CSV: {len(df_emp):,}")
logger.info(f"Tickers detectados: {len(tickers_raw):,}")

def ric_candidates(t: str) -> list[str]:
    """Arma candidatos de RIC para equity según convención de mercado."""
    # Si ya viene con sufijo (ej. AAPL.O), se usa tal cual
    if "." in t:
        return [t]
    # Si no tiene sufijo, se prueban .N y .O
    return [f"{t}.N", f"{t}.O"]


all_frames: list[pd.DataFrame] = []
success = 0
fail = 0

for t in tickers_raw:
    got_it = False

    # Probamos candidatos por ticker
    for ric in ric_candidates(t):
        try:
            ts = ek.get_timeseries(
                ric,
                fields=["CLOSE", "VOLUME"],
                start_date=START_DATE,
                end_date=END_DATE,
                interval="daily",
            )

            # Si trajo datos, normalizamos y guardamos
            if ts is not None and not ts.empty:
                ts = ts.reset_index().rename(columns={"Date": "date"})
                ts["ric"] = ric

                ts["date"] = pd.to_datetime(ts["date"], errors="coerce")
                if getattr(ts["date"].dt, "tz", None) is not None:
                    ts["date"] = ts["date"].dt.tz_localize(None)

                all_frames.append(ts[["date", "ric", "CLOSE", "VOLUME"]])

                dmin = ts["date"].min()
                dmax = ts["date"].max()
                logger.info(f"OK {ric:<10} | {dmin.date()} -> {dmax.date()}")

                success += 1
                got_it = True
                break

        except Exception as e:
            logger.warning(f"Error con {ric}: {type(e).__name__}: {e}")

        # Pausa para rate limits
        sleep_request(0.6)

    if not got_it:
        logger.warning(f"Sin datos para ticker: {t}")
        fail += 1

logger.info(f"RICs con datos: {success:,}")
logger.info(f"RICs sin datos: {fail:,}")

# Consolidación final
if all_frames:
    out = pd.concat(all_frames, ignore_index=True)
else:
    out = pd.DataFrame(columns=["date", "ric", "CLOSE", "VOLUME"])

out = out.sort_values(["ric", "date"]).reset_index(drop=True)

# Export (usa nombres definidos en Config si los tenés ahí)
out_parquet = OUT.get("equity_prices_daily", RAW_DIR / "equity_prices_daily.parquet")
out_csv = out_parquet.with_suffix(".csv")

safe_write_parquet(out, out_parquet, index=False)
out.to_csv(out_csv, index=False)
logger.info(f"Output -> {out_csv.as_posix()}")
logger.info(f"Total filas: {len(out):,}")

# Chequeo: series que arrancan tarde
late = out.groupby("ric")["date"].min().reset_index()
late = late[late["date"] > pd.Timestamp("2016-01-01")]

if not late.empty:
    logger.warning("Series que arrancan tarde (min date > 2016-01-01):")
    logger.warning(late.sort_values("date").head(50).to_string(index=False))
else:
    logger.info("Cobertura temporal OK (no hay series que arranquen tarde según el umbral).")

2026-03-19 02:24:04 | INFO | Empresas en CSV: 32
2026-03-19 02:24:04,643 P[2664] [MainThread 16652] Empresas en CSV: 32
2026-03-19 02:24:04 | INFO | Tickers detectados: 32
2026-03-19 02:24:04,646 P[2664] [MainThread 16652] Tickers detectados: 32
2026-03-19 02:24:05 | INFO | OK MCD.N      | 2015-01-02 -> 2025-12-31
2026-03-19 02:24:05,485 P[2664] [MainThread 16652] OK MCD.N      | 2015-01-02 -> 2025-12-31
2026-03-19 02:24:06 | INFO | OK PG.N       | 2015-01-02 -> 2025-12-31
2026-03-19 02:24:06,038 P[2664] [MainThread 16652] OK PG.N       | 2015-01-02 -> 2025-12-31
2026-03-19 02:24:06 | INFO | OK PEP.N      | 2018-04-23 -> 2025-12-31
2026-03-19 02:24:06,790 P[2664] [MainThread 16652] OK PEP.N      | 2018-04-23 -> 2025-12-31
2026-03-19 02:24:07 | INFO | OK KO.N       | 2015-01-02 -> 2025-12-31
2026-03-19 02:24:07,257 P[2664] [MainThread 16652] OK KO.N       | 2015-01-02 -> 2025-12-31
2026-03-19 02:24:07 | INFO | OK WMT.N      | 2025-12-09 -> 2025-12-31
2026-03-19 02:24:07,549 P[2664] [Mai

## 9) Equity — retornos logarítmicos (diarios)

**Input:** `data/raw/equity_prices_daily.parquet` (date, ric, CLOSE, VOLUME).  
**Output:** `data/clean/equity_returns_daily.parquet` (+ `.csv`).  
**Chequeos:** filas de entrada, cantidad de RICs, filas con retorno válido, cantidad de RICs con retornos.  
**Notas:** el retorno se calcula como `ln(P_t) - ln(P_{t-1})` por activo.

In [18]:
# ======================
# EQUITY -> retornos log diarios
# ======================

# Si venís del bloque anterior, `out` está en memoria.
# Si preferís hacerlo independiente, descomentá esta lectura.
# out = pd.read_parquet(OUT.get("equity_prices_daily", RAW_DIR / "equity_prices_daily.parquet"))

# Orden + tipo numérico
out = out.sort_values(["ric", "date"]).reset_index(drop=True).copy()
out["CLOSE"] = pd.to_numeric(out["CLOSE"], errors="coerce")

logger.info(f"Filas en precios (entrada): {len(out):,}")
logger.info(f"RICs únicos en precios: {out['ric'].nunique():,}")

# Retorno log: ln(P_t) - ln(P_{t-1}) por ric
out["ret_log"] = (
    out.groupby("ric")["CLOSE"]
       .transform(lambda s: np.log(s).diff())
)

# Se eliminan las primeras observaciones sin retorno (NaN por diff)
out_ret = out.dropna(subset=["ret_log"]).copy()

logger.info(f"Filas con ret_log válido: {len(out_ret):,}")
logger.info(f"RICs con al menos un retorno: {out_ret['ric'].nunique():,}")

# Export
out_parquet = OUT.get("equity_returns_daily", CLEAN_DIR / "equity_returns_daily.parquet")
out_csv = out_parquet.with_suffix(".csv")

safe_write_parquet(out_ret, out_parquet, index=False)
out_ret.to_csv(out_csv, index=False)
logger.info(f"Output -> {out_csv.as_posix()}")

2026-03-19 02:24:21 | INFO | Filas en precios (entrada): 79,168
2026-03-19 02:24:21,475 P[2664] [MainThread 16652] Filas en precios (entrada): 79,168


2026-03-19 02:24:21 | INFO | RICs únicos en precios: 32
2026-03-19 02:24:21,481 P[2664] [MainThread 16652] RICs únicos en precios: 32
c:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\venv\Lib\site-packages\pandas\core\arrays\masked.py:644: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs2, **kwargs)
2026-03-19 02:24:21 | INFO | Filas con ret_log válido: 79,135
2026-03-19 02:24:21,524 P[2664] [MainThread 16652] Filas con ret_log válido: 79,135
2026-03-19 02:24:21 | INFO | RICs con al menos un retorno: 32
2026-03-19 02:24:21,530 P[2664] [MainThread 16652] RICs con al menos un retorno: 32
2026-03-19 02:24:21 | INFO | Output -> C:/Users/vidar/OneDrive/Desktop/UDESA/Tesis/Tesis-main/data/clean/equity_returns_daily.parquet | rows=79,135 cols=5
2026-03-19 02:24:21,588 P[2664] [MainThread 16652] Output -> C:/Users/vidar/OneDrive/Desktop/UDESA/Tesis/Tesis-main/data/clean/equity_returns_daily.parquet | rows=79,135 cols=5
2026-03-19 02:24:21 | INFO

## 10) Mercado (S&P 500) — serie diaria y merge con retornos de equity

**Input:** candidatos de mercado (TR / índice / ETFs) vía Refinitiv; `data/clean/equity_returns_daily.parquet` (date, ric, ret_log).  
**Output:** `data/raw/market_sp500_daily.parquet` y `data/clean/equity_market_returns_daily.parquet`.  
**Chequeos:** fuente elegida, rango de fechas, filas del merge y cobertura por fecha.  
**Notas:** se prueban varias fuentes por disponibilidad/permisos; se calcula retorno log diario del mercado.

In [19]:
# ======================
# MERCADO S&P 500 -> serie diaria + merge con retornos de equity
# ======================

# Candidatos de mercado (orden de preferencia)
MARKET_CANDIDATES = [".SPXT", ".SPX", "SPY.P", "IVV.P", "VOO.P"]

def fetch_market_series(candidates: list[str]) -> tuple[str, pd.DataFrame]:
    """Intenta descargar una serie diaria de mercado y devuelve (ric_elegido, df)."""
    last_err: Optional[Exception] = None

    for ric in candidates:
        try:
            ts = ek.get_timeseries(
                ric,
                fields=["CLOSE"],
                start_date=START_DATE,
                end_date=END_DATE,
                interval="daily",
            )

            if ts is not None and not ts.empty:
                logger.info(f"Fuente de mercado elegida: {ric}")
                return ric, ts

        except Exception as e:
            last_err = e
            logger.warning(f"Intento fallido con {ric}: {type(e).__name__}: {e}")

        # Pausa suave entre intentos
        sleep_request(REQUEST_PAUSE_SEC)

    raise RuntimeError(
        "No se pudo descargar ninguna serie de mercado. "
        f"Último error: {type(last_err).__name__}: {last_err}" if last_err else
        "No se pudo descargar ninguna serie de mercado."
    )


chosen_ric, mkt_ts = fetch_market_series(MARKET_CANDIDATES)

# Normalización + retorno log del mercado
mkt = (
    mkt_ts.rename_axis(index="date")
          .reset_index()
          .rename(columns={"CLOSE": "mkt_close"})
)

mkt["date"] = pd.to_datetime(mkt["date"], errors="coerce")
if getattr(mkt["date"].dt, "tz", None) is not None:
    mkt["date"] = mkt["date"].dt.tz_localize(None)

mkt = mkt.sort_values("date").reset_index(drop=True)
mkt["mkt_close"] = pd.to_numeric(mkt["mkt_close"], errors="coerce")
mkt["mkt_ret_log"] = np.log(mkt["mkt_close"]).diff()

# Guardado de mercado
mkt_parquet = OUT.get("market_prices_daily", RAW_DIR / "market_sp500_daily.parquet")
mkt_csv = mkt_parquet.with_suffix(".csv")

safe_write_parquet(mkt, mkt_parquet, index=False)
mkt.to_csv(mkt_csv, index=False)
logger.info(f"Output -> {mkt_csv.as_posix()}")

# Merge equity + mercado (retornos)
eq_ret_path = OUT.get("equity_returns_daily", CLEAN_DIR / "equity_returns_daily.parquet")
eq_rets = pd.read_parquet(eq_ret_path)[["date", "ric", "ret_log"]].copy()

eq_rets["date"] = pd.to_datetime(eq_rets["date"], errors="coerce")
if getattr(eq_rets["date"].dt, "tz", None) is not None:
    eq_rets["date"] = eq_rets["date"].dt.tz_localize(None)

eq_mkt = (
    eq_rets.merge(mkt[["date", "mkt_ret_log"]], on="date", how="inner")
           .dropna(subset=["ret_log", "mkt_ret_log"])
           .sort_values(["ric", "date"])
           .reset_index(drop=True)
)

logger.info(f"Merge equity+mercado | filas={len(eq_mkt):,} | rics={eq_mkt['ric'].nunique():,}")

eq_mkt_parquet = OUT.get("equity_market_returns_daily", CLEAN_DIR / "equity_market_returns_daily.parquet")
safe_write_parquet(eq_mkt, eq_mkt_parquet, index=False)

2026-03-19 02:24:22,288 P[2664] [MainThread 16652] Error with .SPXT: The user does not have permission for the requested data
2026-03-19 02:24:22,288 P[2664] [MainThread 16652] .SPXT: The user does not have permission for the requested data | 
2026-03-19 02:24:22 | WARNING | Intento fallido con .SPXT: EikonError: Error code -1 | .SPXT: The user does not have permission for the requested data | 
2026-03-19 02:24:22,288 P[2664] [MainThread 16652] Intento fallido con .SPXT: EikonError: Error code -1 | .SPXT: The user does not have permission for the requested data | 
2026-03-19 02:24:22,934 P[2664] [MainThread 16652] Error with .SPX: The user does not have permission for the requested data
2026-03-19 02:24:22,934 P[2664] [MainThread 16652] .SPX: The user does not have permission for the requested data | 
2026-03-19 02:24:22 | WARNING | Intento fallido con .SPX: EikonError: Error code -1 | .SPX: The user does not have permission for the requested data | 
2026-03-19 02:24:22,934 P[2664] [Ma

## 11) ETFs sectoriales — precios diarios (CLOSE)

**Input:** lista de ETFs sectoriales (RICs) vía Refinitiv.  
**Output:** `data/raw/sector_etfs_daily.parquet` (+ `.csv`).  
**Chequeos:** ETFs con datos vs sin datos, rango de fechas por ETF, cantidad de filas por serie.  
**Notas:** se aplican reintentos con backoff ante rate limits (HTTP 429) y pausas entre requests.

In [20]:
# ======================
# ETFs sectoriales -> precios diarios (CLOSE)
# ======================

# Lista de ETFs sectoriales (usar RICs si ya los tenés definidos así)
SECTOR_ETFS = [
    "XLF",     # financials
    "XLE",     # energy
    "XLK",     # tech
    "XLY",     # consumer discretionary
    "XLP",     # consumer staples
    "XLI",     # industrials
    "XLV",     # health care
    "XLU",     # utilities
    "XLRE.K",  # real estate
    "XLB",     # materials
    "XLC",     # communication services
]

def get_ts_safe(
    ric: str,
    start_date: str,
    end_date: str,
    fields: list[str] | None = None,
    max_attempts: int = 3,
) -> Optional[pd.DataFrame]:
    """
    Descarga una timeseries diaria con reintentos si aparece rate limit (429).
    Devuelve DataFrame o None si no se pudo.
    """
    if fields is None:
        fields = ["CLOSE"]

    for attempt in range(1, max_attempts + 1):
        try:
            ts = ek.get_timeseries(
                ric,
                fields=fields,
                start_date=start_date,
                end_date=end_date,
                interval="daily",
            )
            return ts

        except Exception as e:
            msg = str(e)

            # Si es 429, hacemos backoff y reintentamos
            if "429" in msg or "Too many requests" in msg:
                wait_sec = int(RETRY_BACKOFF_SEC * attempt * 2)
                logger.warning(f"429 para {ric} | espero {wait_sec}s | intento {attempt}/{max_attempts}")
                time.sleep(wait_sec)
                continue

            # Otros errores: se registra y se corta
            logger.warning(f"Error con {ric}: {type(e).__name__}: {e}")
            return None

    return None


sec_frames: list[pd.DataFrame] = []
missing: list[str] = []

logger.info(f"Descarga ETFs sectoriales | total={len(SECTOR_ETFS)}")

for ric in SECTOR_ETFS:
    logger.info(f"ETF sector: {ric}")

    ts = get_ts_safe(ric, START_DATE, END_DATE, fields=["CLOSE"], max_attempts=RETRY_MAX)

    if ts is None or ts.empty:
        logger.warning(f"Sin datos: {ric}")
        missing.append(ric)
        sleep_request(0.5)
        continue

    # Normalización estándar
    df = ts.reset_index().rename(columns={"Date": "date", "CLOSE": "price"})
    df["sector_ric"] = ric

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    if getattr(df["date"].dt, "tz", None) is not None:
        df["date"] = df["date"].dt.tz_localize(None)

    df["price"] = pd.to_numeric(df["price"], errors="coerce")

    # Log corto con rango
    dmin = df["date"].min()
    dmax = df["date"].max()
    logger.info(f"OK {ric:<6} | {dmin.date()} -> {dmax.date()} | filas={len(df):,}")

    sec_frames.append(df)

    # Pausa corta para no saturar
    sleep_request(0.5)

if not sec_frames:
    raise RuntimeError("No se pudo descargar ningún ETF sectorial (todas las series vinieron vacías).")

sectors_ts = pd.concat(sec_frames, ignore_index=True)

# Guardado
out_parquet = OUT.get("sector_etfs_daily", RAW_DIR / "sector_etfs_daily.parquet")
out_csv = out_parquet.with_suffix(".csv")

safe_write_parquet(sectors_ts, out_parquet, index=False)
sectors_ts.to_csv(out_csv, index=False)
logger.info(f"Output -> {out_csv.as_posix()}")

# Resumen de cobertura
logger.info(f"ETFs con datos: {sectors_ts['sector_ric'].nunique():,}")
logger.info(f"Filas totales: {len(sectors_ts):,}")

if missing:
    logger.warning(f"ETFs sin datos (muestra): {missing[:10]}")

# Tabla rápida de rangos por ETF (útil para QA)
summary = sectors_ts.groupby("sector_ric")["date"].agg(["min", "max", "count"]).sort_values("count", ascending=False)
logger.info("Cobertura por ETF (top):")
logger.info(summary.head(12).to_string())

2026-03-19 02:24:24 | INFO | Descarga ETFs sectoriales | total=11
2026-03-19 02:24:24,539 P[2664] [MainThread 16652] Descarga ETFs sectoriales | total=11
2026-03-19 02:24:24 | INFO | ETF sector: XLF
2026-03-19 02:24:24,541 P[2664] [MainThread 16652] ETF sector: XLF
2026-03-19 02:24:24 | INFO | OK XLF    | 2015-01-02 -> 2025-12-31 | filas=2,766
2026-03-19 02:24:24,979 P[2664] [MainThread 16652] OK XLF    | 2015-01-02 -> 2025-12-31 | filas=2,766
2026-03-19 02:24:25 | INFO | ETF sector: XLE
2026-03-19 02:24:25,482 P[2664] [MainThread 16652] ETF sector: XLE
2026-03-19 02:24:25 | INFO | OK XLE    | 2015-01-02 -> 2025-12-31 | filas=2,766
2026-03-19 02:24:25,983 P[2664] [MainThread 16652] OK XLE    | 2015-01-02 -> 2025-12-31 | filas=2,766
2026-03-19 02:24:26 | INFO | ETF sector: XLK
2026-03-19 02:24:26,486 P[2664] [MainThread 16652] ETF sector: XLK
2026-03-19 02:24:26 | INFO | OK XLK    | 2015-01-02 -> 2025-12-31 | filas=2,766
2026-03-19 02:24:26,823 P[2664] [MainThread 16652] OK XLK    | 201

## 12) Fundamentals — descarga trimestral (Refinitiv)

**Input:** `data/inputs/empresas_tickers_test.csv` (Ticker) + overrides de RIC puntuales.  
**Output:** `data/raw/fundamentals_extended_q.parquet` (+ `.csv`).  
**Chequeos:** RICs con fundamentals vs sin fundamentals; quarters esperados por firma; duplicados por (ric, date).  
**Notas:** se descarga por batches con backoff ante rate limits (HTTP 429). La fecha trimestral se asigna por grilla Q 2015–2025 para uniformizar el panel.

In [21]:
# ======================
# FUNDAMENTALS -> descarga trimestral (Q) por batches
# ======================

import math
import time
from typing import Iterable

# Overrides puntuales de RIC para consultas (casos conocidos)
RIC_OVERRIDE = {
    "PEP":   "PEP.O",
    "ABBV":  "ABBV.K",
    "HON":   "HON.O",
    "AAPL":  "AAPL.O",
    "CSCO":  "CSCO.O",
    "ORCL":  "ORCL.K",
    "META":  "META.O",
    "GOOGL": "GOOGL.O",
    "MSFT":  "MSFT.O",
}

# Fields (fundamentals extendidos)
FUND_FIELDS = [
    "TR.F.TotAssets",
    "TR.F.TotLiab",
    "TR.F.TotShHoldEq",
    "TR.F.TotCurrAssets",
    "TR.F.TotCurrLiab",
    "TR.TotalDebtActValue",
    "TR.F.CashSTInvst",
    "TR.F.STDebtCurrPortOfLTDebt",
    "TR.F.DebtLTTot",
    "TR.F.TotRevenue",
    "TR.F.EBITDA",
    "TR.F.NetCashFlowOp",
    "TR.IntExp",
    "TR.F.CAPEXTot",
]

FUND_PARAMS = {
    "SDate": "2015-01-01",
    "EDate": "2025-12-31",
    "Frq": "Q",
    "Curn": "USD",
}

BATCH_SIZE = 20
SLEEP_BATCH_SEC = 1.0
MAX_RETRY = 3


def chunked(items: list[str], n: int) -> Iterable[list[str]]:
    for i in range(0, len(items), n):
        yield items[i:i + n]


def get_data_safe(
    rics_batch: list[str],
    fields: list[str],
    params: dict,
    max_retry: int = 3
) -> pd.DataFrame:
    """
    Wrapper para ek.get_data con reintentos ante 429.
    Devuelve DataFrame (puede venir vacío).
    """
    for attempt in range(1, max_retry + 1):
        try:
            df, err = ek.get_data(rics_batch, fields=fields, parameters=params)

            if err is not None and len(err):
                logger.warning(f"Refinitiv devolvió warnings/errores (muestra): {str(err[:2])}")

            return df if df is not None else pd.DataFrame()

        except Exception as e:
            msg = str(e)

            if "429" in msg or "Too many requests" in msg:
                wait_sec = 4 + (attempt - 1) * 4
                logger.warning(f"429 en batch | espero {wait_sec}s | intento {attempt}/{max_retry}")
                time.sleep(wait_sec)
                continue

            logger.warning(f"Error en batch (no 429): {type(e).__name__}: {e}")
            return pd.DataFrame()

    return pd.DataFrame()


def to_snake(col: str) -> str:
    x = col.replace("TR.", "").replace("F.", "").replace(":", ".")
    x = x.replace(".", "_")
    return x.lower()


# ----------------------
# 1) Universo + ric_query
# ----------------------
df_emp = pd.read_csv(UNIVERSE_FILE, dtype=str).dropna(subset=["Ticker"]).reset_index(drop=True)

df_emp["ticker"] = df_emp["Ticker"].astype(str).str.strip()
df_emp["ric_query"] = df_emp["ticker"].map(RIC_OVERRIDE).fillna(df_emp["ticker"])

rics = df_emp["ric_query"].dropna().astype(str).unique().tolist()

logger.info(f"Empresas en universo: {len(df_emp):,}")
logger.info(f"RICs únicos (consulta): {len(rics):,}")


# ----------------------
# 2) Descarga por batches
# ----------------------
all_frames: list[pd.DataFrame] = []

logger.info("Descargando fundamentals (Q) por batches...")

for i, batch in enumerate(chunked(rics, BATCH_SIZE), start=1):
    logger.info(f"Batch {i} | size={len(batch)}")

    df_batch = get_data_safe(batch, FUND_FIELDS, FUND_PARAMS, max_retry=MAX_RETRY)
    if not df_batch.empty:
        all_frames.append(df_batch)

    time.sleep(SLEEP_BATCH_SEC)

if not all_frames:
    raise RuntimeError("No se descargaron fundamentals. Revisar fields/permisos/conexión.")

fund = pd.concat(all_frames, ignore_index=True)


# ----------------------
# 3) Normalización de columnas
# ----------------------
if "Instrument" in fund.columns:
    fund = fund.rename(columns={"Instrument": "ric"})
elif "RIC" in fund.columns:
    fund = fund.rename(columns={"RIC": "ric"})

fund.columns = [to_snake(c) for c in fund.columns]


# ----------------------
# 4) Mapeo de rastreo (ticker original)
# ----------------------
df_map = df_emp[["ticker", "ric_query"]].rename(columns={"ric_query": "ric"})
fund = fund.merge(df_map, on="ric", how="left")


# ----------------------
# 5) Fecha trimestral (grilla Q 2015–2025)
# ----------------------
start_q = pd.Timestamp("2015-01-01")
end_q   = pd.Timestamp("2025-12-31")
full_q_range = pd.date_range(start=start_q, end=end_q, freq="Q")


def assign_quarter_grid(g: pd.DataFrame) -> pd.DataFrame:
    g = g.reset_index(drop=True)
    n = len(g)

    if n == 0:
        return g

    reps = math.ceil(n / len(full_q_range))
    dates = list(full_q_range) * reps
    g["date"] = dates[:n]
    return g


fund = fund.groupby("ric", group_keys=False).apply(assign_quarter_grid)

fund = (
    fund.sort_values(["ric", "date"])
        .groupby(["ric", "date"], as_index=False)
        .first()
)


# ----------------------
# 6) QA y export
# ----------------------
logger.info(f"Fundamentals final | filas={len(fund):,} | rics={fund['ric'].nunique():,}")
logger.info(f"Quarters esperados (grilla): {len(full_q_range):,}")

missing = df_emp.loc[
    ~df_emp["ric_query"].isin(fund["ric"]),
    ["Issuer", "ticker", "ric_query"]
]

if not missing.empty:
    logger.warning(f"Empresas sin fundamentals: {len(missing):,}")
    logger.warning(missing.head(20).to_string(index=False))

out_parquet = OUT.get("fundamentals_q", RAW_DIR / "fundamentals_extended_q.parquet")
out_csv = out_parquet.with_suffix(".csv")

safe_write_parquet(fund, out_parquet, index=False)
fund.to_csv(out_csv, index=False)

logger.info(f"Output -> {out_csv.as_posix()}")

2026-03-19 02:24:35 | INFO | Empresas en universo: 32
2026-03-19 02:24:35,545 P[2664] [MainThread 16652] Empresas en universo: 32
2026-03-19 02:24:35 | INFO | RICs únicos (consulta): 32
2026-03-19 02:24:35,545 P[2664] [MainThread 16652] RICs únicos (consulta): 32
2026-03-19 02:24:35 | INFO | Descargando fundamentals (Q) por batches...
2026-03-19 02:24:35,545 P[2664] [MainThread 16652] Descargando fundamentals (Q) por batches...
2026-03-19 02:24:35 | INFO | Batch 1 | size=20
2026-03-19 02:24:35,545 P[2664] [MainThread 16652] Batch 1 | size=20
c:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\venv\Lib\site-packages\pandas\core\dtypes\cast.py:1056: RuntimeWarning: invalid value encountered in cast
  if (arr.astype(int) == arr).all():
c:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\venv\Lib\site-packages\pandas\core\dtypes\cast.py:1080: RuntimeWarning: invalid value encountered in cast
  if (arr.astype(int) == arr).all():
2026-03-19 02:24:40 | INFO | Batch 2 | size=12
2026-03-1

## 13) Curva soberana USD (UST) — yields diarios por tenor

**Input:** RICs de UST (`US1MT=RR`, `US3MT=RR`, …) vía Refinitiv.  
**Output:** `data/raw/ust_yield_curve_daily.parquet` (+ `.csv`).  
**Chequeos:** tenores con datos vs sin datos, rango de fechas por tenor, cantidad de filas.  
**Notas:** se aplican reintentos con backoff ante rate limits (HTTP 429) y pausas entre requests.

In [22]:
# ======================
# UST yield curve -> daily (por tenor)
# ======================

UST_RICS = {
    "US1M":  "US1MT=RR",
    "US3M":  "US3MT=RR",
    "US6M":  "US6MT=RR",
    "US1Y":  "US1YT=RR",
    "US2Y":  "US2YT=RR",
    "US3Y":  "US3YT=RR",
    "US5Y":  "US5YT=RR",
    "US7Y":  "US7YT=RR",
    "US10Y": "US10YT=RR",
    "US30Y": "US30YT=RR",
}

def get_ts_safe(
    ric: str,
    start_date: str,
    end_date: str,
    fields: list[str] | None = None,
    max_attempts: int = 3,
) -> Optional[pd.DataFrame]:
    """Timeseries diaria con reintentos si cae 429. Devuelve DataFrame o None."""
    if fields is None:
        fields = ["CLOSE"]

    for attempt in range(1, max_attempts + 1):
        try:
            ts = ek.get_timeseries(
                ric,
                fields=fields,
                start_date=start_date,
                end_date=end_date,
                interval="daily",
            )
            return ts

        except Exception as e:
            msg = str(e)

            if "429" in msg or "Too many requests" in msg:
                wait_sec = 4 + (attempt - 1) * 4
                logger.warning(f"429 para {ric} | espero {wait_sec}s | intento {attempt}/{max_attempts}")
                time.sleep(wait_sec)
                continue

            logger.warning(f"Error con {ric}: {type(e).__name__}: {e}")
            return None

    return None


logger.info("Descargando curva UST (USD) por tenor...")
logger.info(f"Tenores: {list(UST_RICS.keys())}")

ust_frames: list[pd.DataFrame] = []
missing: list[str] = []

for tenor, ric in UST_RICS.items():
    logger.info(f"UST {tenor} | {ric}")

    ts = get_ts_safe(ric, START_DATE, END_DATE, fields=["CLOSE"], max_attempts=RETRY_MAX)

    if ts is None or ts.empty:
        logger.warning(f"Sin datos: {tenor} ({ric})")
        missing.append(tenor)
        sleep_request(0.4)
        continue

    df = ts.reset_index().rename(columns={"Date": "date", "CLOSE": "yield"})
    df["tenor"] = tenor
    df["ric"] = ric

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    if getattr(df["date"].dt, "tz", None) is not None:
        df["date"] = df["date"].dt.tz_localize(None)

    df["yield"] = pd.to_numeric(df["yield"], errors="coerce")

    dmin = df["date"].min()
    dmax = df["date"].max()
    logger.info(f"OK {tenor:<4} | {dmin.date()} -> {dmax.date()} | filas={len(df):,}")

    ust_frames.append(df)

    # Pausa corta para no saturar
    sleep_request(0.4)

if not ust_frames:
    raise RuntimeError("No se pudo descargar ningún punto de la curva UST.")

ust = pd.concat(ust_frames, ignore_index=True)

# Export
out_parquet = OUT.get("ust_yield_curve_daily", RAW_DIR / "ust_yield_curve_daily.parquet")
out_csv = out_parquet.with_suffix(".csv")

safe_write_parquet(ust, out_parquet, index=False)
ust.to_csv(out_csv, index=False)
logger.info(f"Output -> {out_csv.as_posix()}")

# Resumen corto
logger.info(f"Tenores con datos: {ust['tenor'].nunique():,}")
logger.info(f"Filas totales: {len(ust):,}")
logger.info(f"Rango global: {ust['date'].min().date()} -> {ust['date'].max().date()}")

if missing:
    logger.warning(f"Tenores sin datos: {missing}")

2026-03-19 02:24:44 | INFO | Descargando curva UST (USD) por tenor...
2026-03-19 02:24:44,741 P[2664] [MainThread 16652] Descargando curva UST (USD) por tenor...
2026-03-19 02:24:44 | INFO | Tenores: ['US1M', 'US3M', 'US6M', 'US1Y', 'US2Y', 'US3Y', 'US5Y', 'US7Y', 'US10Y', 'US30Y']
2026-03-19 02:24:44,742 P[2664] [MainThread 16652] Tenores: ['US1M', 'US3M', 'US6M', 'US1Y', 'US2Y', 'US3Y', 'US5Y', 'US7Y', 'US10Y', 'US30Y']
2026-03-19 02:24:44 | INFO | UST US1M | US1MT=RR
2026-03-19 02:24:44,744 P[2664] [MainThread 16652] UST US1M | US1MT=RR
2026-03-19 02:24:45 | INFO | OK US1M | 2015-01-02 -> 2025-12-31 | filas=2,750
2026-03-19 02:24:45,315 P[2664] [MainThread 16652] OK US1M | 2015-01-02 -> 2025-12-31 | filas=2,750
2026-03-19 02:24:45 | INFO | UST US3M | US3MT=RR
2026-03-19 02:24:45,717 P[2664] [MainThread 16652] UST US3M | US3MT=RR
2026-03-19 02:24:46 | INFO | OK US3M | 2015-01-02 -> 2025-12-31 | filas=2,750
2026-03-19 02:24:46,136 P[2664] [MainThread 16652] OK US3M | 2015-01-02 -> 202

## 13bis) OAS por bono — histórico mensual (Refinitiv)


In [23]:
# ======================
# OAS -> histórico mensual por bono
# ======================

# 1) Tomar bond RICs desde el universo ya armado
if "df_bonds_universe" not in globals():
    raise RuntimeError(
        "df_bonds_universe no existe en memoria. "
        "Primero corré la celda que lee los Excel de bonds_empresas."
    )

if "bond_ric" not in df_bonds_universe.columns:
    raise RuntimeError(
        "df_bonds_universe no tiene columna 'bond_ric'. "
        "Revisá la celda de bonos para preservar el RIC del bono."
    )

bonds_full = df_bonds_universe.copy()

bond_rics = (
    bonds_full["bond_ric"]
    .dropna()
    .astype(str)
    .str.strip()
    .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    .dropna()
    .unique()
    .tolist()
)

print(f"Bond RICs para OAS: {len(bond_rics)}")

if len(bond_rics) == 0:
    raise RuntimeError("La lista de bond_rics quedó vacía.")

# 2) Descargar OAS histórico mensual
oas_frames = []

for ric in bond_rics:
    print(f"\nTesting OAS history for {ric}")

    try:
        df, err = ek.get_data(
            [ric],
            ["TR.OPTIONADJUSTEDSPREADBID.date", "TR.OPTIONADJUSTEDSPREADBID"],
            parameters={
                "SDate": START_DATE,
                "EDate": END_DATE,
                "Frq": "M"
            }
        )

        print(df.head())
        print(err)

        if df is not None and len(df) > 0:
            df["source_ric"] = ric
            oas_frames.append(df)

    except Exception as e:
        print(f"Error OAS {ric}: {type(e).__name__}: {e}")

# 3) Consolidar
oas_all = pd.concat(oas_frames, ignore_index=True) if oas_frames else pd.DataFrame()

print("\n=== OAS HIST ALL ===")
print(oas_all.head(20))

if oas_all.empty:
    print("WARNING: No se descargó ningún OAS.")
else:
    # limpieza
    oas_all["Date"] = pd.to_datetime(oas_all["Date"], errors="coerce")
    oas_all["Option Adjusted Spread Bid"] = pd.to_numeric(
        oas_all["Option Adjusted Spread Bid"], errors="coerce"
    )

    oas_all = oas_all.rename(columns={
        "source_ric": "bond_ric",
        "Date": "date",
        "Option Adjusted Spread Bid": "oas_bps"
    })

    # enrich con issuer / ticker
    oas_all = oas_all.merge(
        bonds_full[["Issuer", "Ticker", "bond_ric"]].drop_duplicates(),
        on="bond_ric",
        how="left"
    )

    oas_all = oas_all.sort_values(["bond_ric", "date"]).reset_index(drop=True)

    print("\n=== OAS COVERAGE ===")
    print(
        oas_all.groupby("bond_ric")
        .agg(
            first_date=("date", "min"),
            last_date=("date", "max"),
            n_obs=("oas_bps", "count")
        )
        .reset_index()
    )

    safe_write_parquet(
        oas_all,
        OUT["oas_spreads_monthly_bond"],
        index=False
    )

    print(f"\nGuardado OAS en: {OUT['oas_spreads_monthly_bond']}")
    print(f"Filas OAS: {len(oas_all)}")
    print(f"Bond RICs con OAS: {oas_all['bond_ric'].nunique()} / {len(bond_rics)}")

Bond RICs para OAS: 276

Testing OAS history for 580135BY6=
   Instrument                  Date  Option Adjusted Spread Bid
0  580135BY6=  2015-01-30T00:00:00Z                       123.7
1  580135BY6=  2015-02-27T00:00:00Z                       130.7
2  580135BY6=  2015-03-31T00:00:00Z                       132.8
3  580135BY6=  2015-04-30T00:00:00Z                       133.7
4  580135BY6=  2015-05-29T00:00:00Z                       136.2
None

Testing OAS history for 742718FZ7=
   Instrument  Date  Option Adjusted Spread Bid
0  742718FZ7=  <NA>                        <NA>
[{'code': 416, 'col': 1, 'message': "Unable to collect data for the field 'TR.OPTIONADJUSTEDSPREADBID.DATE' and some specific identifier(s).", 'row': 0}, {'code': 416, 'col': 2, 'message': "Unable to collect data for the field 'TR.OPTIONADJUSTEDSPREADBID' and some specific identifier(s).", 'row': 0}]

Testing OAS history for 713448GA0=
   Instrument                  Date  Option Adjusted Spread Bid
0  713448GA0=  20

2026-03-19 03:08:16 | INFO | Output -> C:/Users/vidar/OneDrive/Desktop/UDESA/Tesis/Tesis-main/data/raw/oas_spreads_monthly_bond.parquet | rows=11,719 cols=6
2026-03-19 03:08:16,954 P[2664] [MainThread 16652] Output -> C:/Users/vidar/OneDrive/Desktop/UDESA/Tesis/Tesis-main/data/raw/oas_spreads_monthly_bond.parquet | rows=11,719 cols=6


   Instrument                  Date  Option Adjusted Spread Bid
0  26441CBL8=  2021-06-30T00:00:00Z                       101.0
1  26441CBL8=  2021-07-30T00:00:00Z                        96.8
2  26441CBL8=  2021-08-31T00:00:00Z                        95.5
3  26441CBL8=  2021-09-30T00:00:00Z                        98.1
4  26441CBL8=  2021-10-29T00:00:00Z                       102.6
None

=== OAS HIST ALL ===
    Instrument                  Date  Option Adjusted Spread Bid  source_ric
0   580135BY6=  2015-01-30T00:00:00Z                       123.7  580135BY6=
1   580135BY6=  2015-02-27T00:00:00Z                       130.7  580135BY6=
2   580135BY6=  2015-03-31T00:00:00Z                       132.8  580135BY6=
3   580135BY6=  2015-04-30T00:00:00Z                       133.7  580135BY6=
4   580135BY6=  2015-05-29T00:00:00Z                       136.2  580135BY6=
5   580135BY6=  2015-06-30T00:00:00Z                       134.7  580135BY6=
6   580135BY6=  2015-07-31T00:00:00Z              

# 13bis enriched

In [7]:
# ==========================================================
# 13bis. Enriquecer bonos con YTM y residual maturity
# ==========================================================

import pandas as pd
import numpy as np

# ----------------------------------------------------------
# Paths
# ----------------------------------------------------------
YIELDS_DIR = RAW_DIR / "bonds_empresas" / "Yields"

OUT_PARQUET = CLEAN_DIR / "bonds_monthly_spreads.parquet"
OUT_CSV     = CLEAN_DIR / "bonds_monthly_spreads.csv"

# ----------------------------------------------------------
# Helpers
# ----------------------------------------------------------
def to_eom(s):
    return pd.to_datetime(s, errors="coerce").dt.to_period("M").dt.to_timestamp("M")

def pick_col(df, candidates):
    cols = list(df.columns)
    lower_map = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand in cols:
            return cand
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None

def clean_ric(s):
    s = s.astype(str).str.strip()
    s = s.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    return s

# ----------------------------------------------------------
# 1) Validaciones básicas
# ----------------------------------------------------------
if "df_bonds_universe" not in globals():
    raise ValueError(
        "No encontré 'df_bonds_universe' en memoria. "
        "Primero corré la celda donde se leen los excels de bonds_empresas."
    )

oas = pd.read_parquet(RAW_DIR / "oas_spreads_monthly_bond.parquet")
bonds_meta = df_bonds_universe.copy()

# ----------------------------------------------------------
# 2) Normalizar metadata de bonos
# ----------------------------------------------------------
ric_col = pick_col(bonds_meta, ["bond_ric", "Preferred RIC", "RIC", "Instrument", "Issue RIC"])
if ric_col is None:
    raise ValueError(
        f"No encontré columna de RIC en df_bonds_universe. Columnas: {list(bonds_meta.columns)}"
    )

mat_col = pick_col(bonds_meta, ["Maturity", "Maturity Date", "maturity"])
if mat_col is None:
    raise ValueError(
        f"No encontré columna de Maturity en df_bonds_universe. Columnas: {list(bonds_meta.columns)}"
    )

issuer_col = pick_col(bonds_meta, ["Issuer"])
ticker_col = pick_col(bonds_meta, ["Ticker"])
isin_col   = pick_col(bonds_meta, ["ISIN"])
coupon_col = pick_col(bonds_meta, ["Coupon"])

keep_meta = [ric_col, mat_col]
for c in [issuer_col, ticker_col, isin_col, coupon_col]:
    if c is not None and c not in keep_meta:
        keep_meta.append(c)

bonds_meta = bonds_meta[keep_meta].copy()
bonds_meta = bonds_meta.rename(columns={
    ric_col: "bond_ric",
    mat_col: "Maturity",
    issuer_col if issuer_col is not None else ric_col: issuer_col if issuer_col is not None else ric_col,
    ticker_col if ticker_col is not None else ric_col: ticker_col if ticker_col is not None else ric_col,
    isin_col if isin_col is not None else ric_col: isin_col if isin_col is not None else ric_col,
    coupon_col if coupon_col is not None else ric_col: coupon_col if coupon_col is not None else ric_col,
})

bonds_meta["bond_ric"] = clean_ric(bonds_meta["bond_ric"])
bonds_meta["Maturity"] = pd.to_datetime(bonds_meta["Maturity"], errors="coerce")

if "Coupon" in bonds_meta.columns:
    bonds_meta["Coupon"] = pd.to_numeric(bonds_meta["Coupon"], errors="coerce")

bonds_meta = bonds_meta.dropna(subset=["bond_ric"]).drop_duplicates(subset=["bond_ric"])

print("\n=== QA bonds_meta ===")
print("Shape:", bonds_meta.shape)
print("RICs únicos:", bonds_meta["bond_ric"].nunique())
print("NA Maturity:", round(bonds_meta["Maturity"].isna().mean(), 4))
show_cols = [c for c in ["Issuer", "Ticker", "bond_ric", "Maturity", "ISIN", "Coupon"] if c in bonds_meta.columns]
print(bonds_meta[show_cols].head())

# ----------------------------------------------------------
# 3) Leer yields por bono
# ----------------------------------------------------------
yield_files = sorted(YIELDS_DIR.glob("*.xlsx"))
if not yield_files:
    raise FileNotFoundError(f"No encontré archivos .xlsx en {YIELDS_DIR}")

frames = []
for fp in yield_files:
    try:
        tmp = pd.read_excel(fp)
        tmp.columns = [str(c).strip() for c in tmp.columns]

        ric_y = pick_col(tmp, ["RIC", "Instrument", "bond_ric", "Preferred RIC", "Issue RIC"])
        date_y = pick_col(tmp, ["Date", "date"])
        ytm_y = pick_col(tmp, ["Yield to Maturity", "YTM", "ytm", "Mid Yield", "Yield"])

        if ric_y is None or date_y is None or ytm_y is None:
            print(f"⚠️ Salteo {fp.name}: no encontré columnas necesarias")
            print("   Columnas:", list(tmp.columns))
            continue

        tmp = tmp[[ric_y, date_y, ytm_y]].copy()
        tmp = tmp.rename(columns={
            ric_y: "bond_ric",
            date_y: "date",
            ytm_y: "ytm"
        })

        tmp["bond_ric"] = clean_ric(tmp["bond_ric"])
        tmp["date"] = to_eom(tmp["date"])
        tmp["ytm"] = pd.to_numeric(tmp["ytm"], errors="coerce")
        tmp["source_file"] = fp.name

        frames.append(tmp)

    except Exception as e:
        print(f"⚠️ Error leyendo {fp.name}: {e}")

if not frames:
    raise ValueError("No pude construir bond_yields desde los excels de Yields.")

bond_yields = pd.concat(frames, ignore_index=True)
bond_yields = bond_yields.dropna(subset=["bond_ric", "date"])
bond_yields = bond_yields.sort_values(["bond_ric", "date"])

# si hay duplicados bond_ric-date, promedia
bond_yields = (
    bond_yields
    .groupby(["bond_ric", "date"], as_index=False)
    .agg(
        ytm=("ytm", "mean"),
        n_obs=("ytm", "size")
    )
)

print("\n=== QA bond_yields ===")
print("Shape:", bond_yields.shape)
print("RICs únicos:", bond_yields["bond_ric"].nunique())
print("NA ytm:", round(bond_yields["ytm"].isna().mean(), 4))
print(bond_yields.head())

# ----------------------------------------------------------
# 4) Merge yields + metadata y residual maturity
# ----------------------------------------------------------
bond_yields = bond_yields.merge(
    bonds_meta,
    on="bond_ric",
    how="left",
    validate="m:1"
)

bond_yields["residual_maturity_years"] = (
    (bond_yields["Maturity"] - bond_yields["date"]).dt.days / 365.25
)

# filtrar residuos absurdos
bond_yields.loc[bond_yields["residual_maturity_years"] < 0, "residual_maturity_years"] = np.nan

print("\n=== QA yields + maturity ===")
print("Shape:", bond_yields.shape)
print("NA Maturity post-merge:", round(bond_yields["Maturity"].isna().mean(), 4))
print("NA residual_maturity_years:", round(bond_yields["residual_maturity_years"].isna().mean(), 4))
show_cols = [c for c in ["bond_ric", "date", "ytm", "Maturity", "residual_maturity_years", "Issuer", "Ticker"] if c in bond_yields.columns]
print(bond_yields[show_cols].head())

# ----------------------------------------------------------
# 5) Preparar OAS mensual por bono
# ----------------------------------------------------------
oas.columns = [str(c).strip() for c in oas.columns]

ric_o = pick_col(oas, ["bond_ric", "RIC", "Instrument", "Preferred RIC", "Issue RIC"])
date_o = pick_col(oas, ["date", "Date"])
oas_o  = pick_col(oas, ["oas", "OAS", "oas_bps", "OAS(bp)", "option_adjusted_spread"])

if ric_o is None or date_o is None:
    raise ValueError(
        f"No encontré columnas mínimas en oas_spreads_monthly_bond. Columnas: {list(oas.columns)}"
    )

oas = oas.rename(columns={
    ric_o: "bond_ric",
    date_o: "date",
})

if oas_o is not None and oas_o != "oas":
    oas = oas.rename(columns={oas_o: "oas"})

oas["bond_ric"] = clean_ric(oas["bond_ric"])
oas["date"] = to_eom(oas["date"])

if "oas" in oas.columns:
    oas["oas"] = pd.to_numeric(oas["oas"], errors="coerce")

# si hubiera múltiples filas por bond_ric-date, colapsa
agg_dict = {}
for c in oas.columns:
    if c in ["bond_ric", "date"]:
        continue
    if c == "oas":
        agg_dict[c] = "mean"
    else:
        agg_dict[c] = "first"

oas_m = oas.groupby(["bond_ric", "date"], as_index=False).agg(agg_dict)

print("\n=== QA oas_m ===")
print("Shape:", oas_m.shape)
print("RICs únicos:", oas_m["bond_ric"].nunique())
print(oas_m.head())

# ----------------------------------------------------------
# 6) Merge final: OAS + YTM + residual maturity
# ----------------------------------------------------------
merge_cols = ["bond_ric", "date", "ytm", "residual_maturity_years"]
extra_cols = [c for c in ["Issuer", "Ticker", "ISIN", "Coupon", "Maturity"] if c in bond_yields.columns]
merge_cols = merge_cols + extra_cols

bonds_monthly_spreads = oas_m.merge(
    bond_yields[merge_cols],
    on=["bond_ric", "date"],
    how="left"
)

print("\n=== QA bonds_monthly_spreads ===")
print("Shape:", bonds_monthly_spreads.shape)
print("NA ytm:", round(bonds_monthly_spreads["ytm"].isna().mean(), 4))
print("NA residual_maturity_years:", round(bonds_monthly_spreads["residual_maturity_years"].isna().mean(), 4))

matched = bonds_monthly_spreads["ytm"].notna().mean()
print("Match rate YTM:", round(matched, 4))

# ----------------------------------------------------------
# 7) Guardar
# ----------------------------------------------------------
OUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)

bonds_monthly_spreads.to_parquet(OUT_PARQUET, index=False)
bonds_monthly_spreads.to_csv(OUT_CSV, index=False)

print(f"\n✅ Guardado parquet: {OUT_PARQUET}")
print(f"✅ Guardado csv:     {OUT_CSV}")


=== QA bonds_meta ===
Shape: (276, 6)
RICs únicos: 276
NA Maturity: 0.0
                Issuer Ticker    bond_ric            Maturity          ISIN  \
0      McDonald's Corp    MCD  580135BY6= 2028-01-07 23:59:12  US580135BY65   
1  Procter & Gamble Co     PG  742718FZ7= 2028-01-25 23:59:12  US742718FZ79   
2          PepsiCo Inc    PEP  713448GA0= 2028-02-06 23:59:12  US713448GA00   
3          PepsiCo Inc    PEP  713448FL7= 2028-02-17 23:59:12  US713448FL73   
4         Coca-Cola Co     KO  191216DJ6= 2028-03-04 23:59:12  US191216DJ60   

   Coupon  
0   6.375  
1   3.950  
2   4.450  
3   3.600  
4   1.500  
⚠️ Error leyendo ~$Yields_energy.xlsx: Excel file format cannot be determined, you must specify an engine manually.
⚠️ Error leyendo ~$Yields_technology.xlsx: Excel file format cannot be determined, you must specify an engine manually.

=== QA bond_yields ===
Shape: (14172, 4)
RICs únicos: 267
NA ytm: 0.0001
     bond_ric       date       ytm  n_obs
0  00287YBW8= 2019-11-30  3.

C:\Users\vidar\AppData\Local\Temp\ipykernel_20384\263096078.py:20: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  return pd.to_datetime(s, errors="coerce").dt.to_period("M").dt.to_timestamp("M")


## 14) CDS corporativo — spreads diarios (Primary CDS RIC)

**Input:** `data/inputs/empresas_tickers_test.csv` (Ticker/Issuer).  
**Output:** `data/raw/cds_spreads_daily.parquet` (+ `.csv`).  
**Chequeos:** emisores con CDS vs sin CDS, campo usado (5Y o genérico), rango de fechas y filas totales.  
**Notas:** se intenta primero `TR.CDSMidSpread5Y`; si no hay cobertura, se usa `TR.CDSMidSpread`. Se aplican pausas y reintentos ante 429.

In [8]:
 # ======================
# CDS -> spreads diarios (Primary CDS RIC + PARMIDSPREAD)
# ======================

def get_primary_cds_ric_batch(rics_batch):
    """
    Devuelve el primary CDS RIC por emisor.
    """
    try:
        df, err = ek.get_data(rics_batch, ["TR.CDSPrimaryCDSRic"])

        print("\nPrimary CDS batch input:", rics_batch[:5])
        print(df.head() if df is not None else "df=None")
        print(err)

        if df is None or df.empty:
            return pd.DataFrame()

        df = df.rename(columns={
            "Instrument": "ric",
            "Primary CDS RIC": "primary_cds_ric"
        }).copy()

        return df

    except Exception as e:
        print(f"Error en primary CDS batch: {type(e).__name__}: {e}")
        return pd.DataFrame()


def get_cds_history_one(cds_ric, start_date, end_date):
    """
    Descarga histórico diario de PAR MID SPREAD para un CDS RIC.
    """
    try:
        df, err = ek.get_data(
            [cds_ric],
            ["TR.CDSType", "TR.PARMIDSPREAD.date", "TR.PARMIDSPREAD"],
            {
                "SDate": start_date,
                "EDate": end_date,
                "DateType": "AD",
                "CURN": "USD"
            }
        )

        print(f"\nTesting CDS history for {cds_ric}")
        print(df.head() if df is not None else "df=None")
        print(err)

        if df is None or df.empty:
            return pd.DataFrame()

        df = df.rename(columns={
            "Instrument": "cds_ric",
            "Date": "date",
            "Par Mid Spread": "cds_spread_bps",
            "CDS Type": "cds_type"
        }).copy()

        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df["cds_spread_bps"] = pd.to_numeric(df["cds_spread_bps"], errors="coerce")

        return df

    except Exception as e:
        print(f"Error CDS {cds_ric}: {type(e).__name__}: {e}")
        return pd.DataFrame()


# ----------------------
# Helper local
# ----------------------

def ticker_to_ric(ticker: str):
    ticker = str(ticker).strip().upper()

    nasdaq = {"AAPL","MSFT","GOOGL","META","CSCO","PEP","HON"}
    nyse   = {"JPM","BAC","C","WFC","GS","MS","XOM","CVX","COP","OXY",
              "JNJ","PFE","MRK","ABBV","PG","KO","MCD","WMT",
              "GE","MMM","CAT","DUK","NEE","SO","ORCL"}

    if ticker in nasdaq:
        return f"{ticker}.O"

    if ticker in nyse:
        return f"{ticker}.N"

    return f"{ticker}.N"


# ----------------------
# 1) Universo de emisores
# ----------------------

meta_path = RAW_DIR / "company_metadata.parquet"

# Universo base con Ticker / Issuer
df_univ = pd.read_csv(UNIVERSE_FILE, dtype=str).copy()
df_univ = df_univ.dropna(subset=["Ticker"]).reset_index(drop=True)

if "Issuer" not in df_univ.columns:
    df_univ["Issuer"] = pd.NA

df_univ["Ticker"] = df_univ["Ticker"].astype(str).str.strip()
df_univ["ric"] = df_univ["Ticker"].apply(ticker_to_ric)

if meta_path.exists():
    meta_emp = pd.read_parquet(meta_path).copy()
    print("Usando metadata de empresas:", meta_path)

    # Normalizar nombre de columna del instrumento
    if "Instrument" in meta_emp.columns:
        meta_emp = meta_emp.rename(columns={"Instrument": "ric"})
    elif "RIC" in meta_emp.columns:
        meta_emp = meta_emp.rename(columns={"RIC": "ric"})

    if "ric" not in meta_emp.columns:
        raise RuntimeError("company_metadata.parquet no tiene columna 'ric'.")

    meta_emp["ric"] = meta_emp["ric"].astype(str).str.strip()

    # Reconstruyo Ticker / Issuer desde el universo original
    df_emp = (
        meta_emp[["ric"]]
        .drop_duplicates()
        .merge(
            df_univ[["Ticker", "Issuer", "ric"]].drop_duplicates(),
            on="ric",
            how="left"
        )
    )

else:
    print("WARNING: no existe company_metadata.parquet. Uso fallback con Ticker.")
    df_emp = df_univ[["Ticker", "Issuer", "ric"]].drop_duplicates().copy()

df_emp["ric"] = df_emp["ric"].astype(str).str.strip()
df_emp = df_emp.dropna(subset=["ric"]).drop_duplicates(subset=["ric"]).reset_index(drop=True)

rics = df_emp["ric"].tolist()
print(f"Emisores a pedir CDS: {len(rics)}")
print("Ejemplos rics:", rics[:10])


# ----------------------
# 2) Primary CDS RIC
# ----------------------
primary_frames = []

for batch in chunked(rics, 25):
    df_batch = get_primary_cds_ric_batch(batch)
    if not df_batch.empty:
        primary_frames.append(df_batch)

primary_cds = pd.concat(primary_frames, ignore_index=True) if primary_frames else pd.DataFrame()

if primary_cds.empty:
    raise RuntimeError("No se pudo obtener ningún Primary CDS RIC.")

primary_cds["primary_cds_ric"] = primary_cds["primary_cds_ric"].astype("string").str.strip()
primary_cds = primary_cds.dropna(subset=["primary_cds_ric"]).drop_duplicates()

print("Primary CDS RICs encontrados:", len(primary_cds))
print(primary_cds.head())


# ----------------------
# 3) Histórico CDS
# ----------------------
frames = []

for _, row in primary_cds.iterrows():
    cds_ric = row["primary_cds_ric"]
    issuer_ric = row["ric"]

    df = get_cds_history_one(cds_ric, START_DATE, END_DATE)

    if df is not None and not df.empty:
        df["ric"] = issuer_ric
        frames.append(df)
        print(f"OK CDS {issuer_ric} -> {cds_ric} | filas={len(df)}")
    else:
        print(f"Sin CDS histórico: {issuer_ric} -> {cds_ric}")

if not frames:
    raise RuntimeError("Ningún emisor devolvió histórico de CDS.")

cds_all = pd.concat(frames, ignore_index=True)

cds_all = cds_all.merge(
    df_emp[["ric", "Issuer", "Ticker"]].drop_duplicates(),
    on="ric",
    how="left"
)

cds_all = cds_all.sort_values(["ric", "date"]).reset_index(drop=True)

safe_write_parquet(cds_all, OUT["cds_spreads_daily"], index=False)

print(f"Filas CDS: {len(cds_all)}")
print(f"Emisores con CDS: {cds_all['ric'].nunique()}")
print("Guardado en:", OUT["cds_spreads_daily"])


Usando metadata de empresas: C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\data\raw\company_metadata.parquet
Emisores a pedir CDS: 32
Ejemplos rics: ['MCD.N', 'PG.N', 'PEP.O', 'KO.N', 'WMT.N', 'OXY.N', 'COP.N', 'XOM.N', 'CVX.N', 'BAC.N']


NameError: name 'chunked' is not defined

## 15) Revenue + market share (SP500) — revenue trimestral y participación por industria

**Input:** universo `0#.SPX` vía Refinitiv + `data/inputs/empresas_tickers_test.csv` para identificar la muestra.  
**Output:**  
- `data/raw/gics_industry_group_sales_quarterly_sp500.csv` (ventas agregadas por industria y trimestre)  
- `data/raw/fundamentals_with_market_share_sp500.csv` (panel SP500)  
- `data/raw/fundamentals_with_market_share.csv` (panel filtrado a la muestra)  

**Chequeos:** fuente de revenue usada, cobertura de GICS, filas por trimestre, market share en [0,1] (cuando hay industria).  
**Notas:** revenue se toma de un set de campos candidatos según disponibilidad/permisos. La agrupación base es `GICS Industry Group` (se puede cambiar a sector/industry si hiciera falta).

In [9]:
# ======================
# Revenue + market share (SP500)
# ======================

# ----------------------
# Helpers de descarga
# ----------------------
def chunked(items: list[str], n: int) -> Iterable[list[str]]:
    """Corta lista en batches."""
    for i in range(0, len(items), n):
        yield items[i:i + n]


def get_data_safe(
    rics_batch: list[str],
    fields: list[str],
    params: dict,
    max_retry: int = 3,
) -> pd.DataFrame:
    """
    Wrapper para ek.get_data con reintentos ante 429.
    Devuelve DataFrame (puede venir vacío).
    """
    for attempt in range(1, max_retry + 1):
        try:
            df, err = ek.get_data(rics_batch, fields=fields, parameters=params)
            if err is not None and len(err):
                logger.warning(f"Refinitiv warnings/errores (muestra): {str(err[:2])}")
            return df if df is not None else pd.DataFrame()

        except Exception as e:
            msg = str(e)
            if "429" in msg or "Too many requests" in msg:
                wait_sec = 4 + (attempt - 1) * 4
                logger.warning(f"429 en batch | espero {wait_sec}s | intento {attempt}/{max_retry}")
                time.sleep(wait_sec)
                continue

            logger.warning(f"Error en batch (no 429): {type(e).__name__}: {e}")
            return pd.DataFrame()

    return pd.DataFrame()


def get_metadata_batched(
    instruments: list[str],
    fields: list[str],
    batch_size: int = 25,
    pause_sec: float = 0.4,
) -> pd.DataFrame:
    """Baja metadata por batches usando ek.get_data (sin parameters)."""
    frames: list[pd.DataFrame] = []

    for i, batch in enumerate(chunked(instruments, batch_size), start=1):
        try:
            df_batch, err = ek.get_data(batch, fields=fields)
            if err is not None and len(err):
                logger.warning(f"Warnings/errores metadata batch (muestra): {str(err[:2])}")
            if df_batch is not None and not df_batch.empty:
                frames.append(df_batch)
        except Exception as e:
            logger.warning(f"Error metadata batch {i} | {type(e).__name__}: {e}")

        time.sleep(pause_sec)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def first_present(cands: list[str], cols_set: set[str]) -> Optional[str]:
    """Devuelve el primer candidato que exista en cols_set."""
    for c in cands:
        if c in cols_set:
            return c
    return None


# ----------------------
# 1) Muestra (para filtrar al final)
# ----------------------
df_emp = pd.read_csv(UNIVERSE_FILE, dtype=str)
df_emp["Ticker"] = df_emp["Ticker"].astype(str).str.strip()

RIC_FIX = {
    "PEP": "PEP.O",
    "ABBV": "ABBV.K",
    "HON": "HON.O",
    "AAPL": "AAPL.O",
    "CSCO": "CSCO.O",
    "ORCL": "ORCL.K",
    "META": "META.O",
    "GOOGL": "GOOGL.O",
    "MSFT": "MSFT.O",
}

def fix_ticker(t: str) -> str:
    t = str(t).strip()
    if not t:
        return t
    base = t.split(".")[0]
    return RIC_FIX.get(base, t)

df_emp["Ticker_fixed"] = df_emp["Ticker"].apply(fix_ticker)
df_emp["ric_base"] = df_emp["Ticker_fixed"].str.split(".").str[0]
target_bases = sorted(df_emp["ric_base"].dropna().unique().tolist())

logger.info(f"Muestra (bases RIC): {len(target_bases):,}")

# ----------------------
# 2) Universo SP500 (0#.SPX)
# ----------------------
logger.info("Descargando universo SP500 (0#.SPX) ...")
spx_df, spx_err = ek.get_data("0#.SPX", ["TR.RIC"])
if spx_err is not None and len(spx_err):
    logger.warning(f"Warnings universo SPX (muestra): {str(spx_err[:3])}")

if spx_df is None or spx_df.empty:
    raise RuntimeError("No llegó universo SPX desde Refinitiv.")

spx_df = spx_df.dropna(subset=["RIC"]).copy()
spx_df["RIC"] = spx_df["RIC"].astype(str).str.strip()
spx_rics = spx_df["RIC"].dropna().unique().tolist()

logger.info(f"RICs SP500 detectados: {len(spx_rics):,}")

# ----------------------
# 3) Metadata GICS SP500
# ----------------------
META_FIELDS = [
    "TR.CommonName",
    "TR.PrimaryRIC",
    "TR.ISIN",
    "TR.CUSIP",
    "TR.PermID",
    "TR.CompanyMarketCap",
    "TR.GICSSector",
    "TR.GICSSectorName",
    "TR.GICSIndustryGroup",
    "TR.GICSIndustryGroupName",
    "TR.GICSIndustry",
    "TR.GICSIndustryName",
    "TR.GICSSubIndustry",
    "TR.GICSSubIndustryName",
    "TR.HeadquartersCountry",
]

logger.info("Descargando metadata (GICS) para SP500 ...")
meta_spx = get_metadata_batched(spx_rics, META_FIELDS, batch_size=25, pause_sec=0.4)

if meta_spx.empty:
    raise RuntimeError("No llegó metadata SPX (GICS).")

meta_spx.columns = [str(c).strip() for c in meta_spx.columns]
meta_spx = meta_spx.rename(columns={
    "Instrument": "ric",
    "GICS Sector Name": "gics_sector_name",
    "GICS Industry Group Name": "gics_industry_group_name",
    "GICS Industry Name": "gics_industry_name",
    "GICS Sub-Industry Name": "gics_subindustry_name",
})

meta_spx["ric"] = meta_spx["ric"].astype(str).str.strip()

gics_cols = [
    "gics_sector_name",
    "gics_industry_group_name",
    "gics_industry_name",
    "gics_subindustry_name",
]

meta_spx_gics = meta_spx[["ric"] + gics_cols].drop_duplicates(subset=["ric"]).copy()
logger.info(f"Metadata GICS SP500 | filas={len(meta_spx_gics):,}")

# ----------------------
# 4) Fundamentals (revenue) SP500
# ----------------------
REV_CANDS = [
    "TR.F.TotRevenue",
    "TR.F.SalesTurnover",
    "TR.F.TotalRevenue",
    "TR.F.SalesOrRevenue",
    "TR.F.NetSales",
    "TR.F.Revenue",
    "TR.F.RevenueReported",
]

fields_funda = REV_CANDS + ["TR.F.PeriodEndDate"]

params = {
    "SDate": "2015-01-01",
    "EDate": "2025-12-31",
    "Frq": "Q",
    "Curn": "USD",
}

logger.info("Descargando fundamentals (revenue) para SP500 ...")

funda_frames: list[pd.DataFrame] = []
BATCH = 20
SLEEP_SEC = 1.0

for i, batch in enumerate(chunked(spx_rics, BATCH), start=1):
    logger.info(f"Batch funda {i} | size={len(batch)}")
    df_b = get_data_safe(batch, fields_funda, params, max_retry=RETRY_MAX)
    if df_b is not None and not df_b.empty:
        funda_frames.append(df_b)
    time.sleep(SLEEP_SEC)

if not funda_frames:
    raise RuntimeError("No llegó fundamentals de revenue.")

funda_all = pd.concat(funda_frames, ignore_index=True)
funda_all.columns = [str(c).strip() for c in funda_all.columns]
cols = set(funda_all.columns)

ric_col = first_present(["Instrument", "RIC", "TR.RIC", "Constituent RIC"], cols)
ped_col = first_present(["TR.F.PeriodEndDate", "Period End Date", "PeriodEndDate"], cols)
rev_col = first_present(
    REV_CANDS + ["Revenue from Business Activities - Total", "Revenue", "Sales/Turnover"],
    cols,
)

missing = [x for x, y in [("ric", ric_col), ("period_end", ped_col), ("revenue", rev_col)] if y is None]
if missing:
    raise RuntimeError(f"Columnas faltantes en fundamentals: {missing} | disponibles={sorted(cols)}")

logger.info(f"Campo revenue usado: {rev_col}")

funda_all = funda_all.rename(columns={ric_col: "ric", ped_col: "period_end", rev_col: "revenue"})
funda_all["ric"] = funda_all["ric"].astype(str).str.strip()
funda_all["period_end"] = pd.to_datetime(funda_all["period_end"], errors="coerce")
funda_all["revenue"] = pd.to_numeric(funda_all["revenue"], errors="coerce")

funda_all = funda_all.dropna(subset=["period_end"]).copy()
logger.info(f"Fundamentals revenue | filas={len(funda_all):,}")

# ----------------------
# 5) Merge + ventas industria + market share
# ----------------------
funda_all = funda_all.merge(meta_spx_gics, on="ric", how="left")

group_col = "gics_industry_group_name"
if group_col not in funda_all.columns:
    raise RuntimeError(f"No existe '{group_col}' en funda_all. Columnas={funda_all.columns.tolist()}")

sector_sales = (
    funda_all.dropna(subset=[group_col])
             .groupby([group_col, "period_end"], as_index=False)["revenue"]
             .sum()
             .rename(columns={"revenue": "industry_sales"})
)

out_sales = OUT.get(
    "gics_industry_group_sales_q_sp500",
    RAW_DIR / "gics_industry_group_sales_quarterly_sp500.csv"
)
sector_sales.to_csv(out_sales, index=False)
logger.info(f"Output -> {out_sales.as_posix()} | filas={len(sector_sales):,}")

funda_all = funda_all.merge(sector_sales, on=[group_col, "period_end"], how="left")
funda_all["market_share"] = funda_all["revenue"] / funda_all["industry_sales"]

# ----------------------
# 6) Export: SP500 completo + muestra
# ----------------------
funda_all["ric_base"] = funda_all["ric"].astype(str).str.split(".").str[0]

out_sp500 = OUT.get("fundamentals_with_market_share_sp500", RAW_DIR / "fundamentals_with_market_share_sp500.csv")
funda_all.to_csv(out_sp500, index=False)
logger.info(f"Output -> {out_sp500.as_posix()} | filas={len(funda_all):,}")

funda_focus = funda_all[funda_all["ric_base"].isin(target_bases)].copy()
out_focus = OUT.get("fundamentals_with_market_share_focus", RAW_DIR / "fundamentals_with_market_share.csv")
funda_focus.to_csv(out_focus, index=False)
logger.info(f"Output -> {out_focus.as_posix()} | filas={len(funda_focus):,}")

# QA rápido: market_share fuera de rango (solo donde hay industria_sales)
bad_ms = funda_all.loc[funda_all["industry_sales"].notna(), "market_share"]
share_oob = ((bad_ms < 0) | (bad_ms > 1)).sum()
if share_oob:
    logger.warning(f"Market share fuera de [0,1] (conteo): {int(share_oob)}")

# ==========================================================
# 7) FIX + Convertir quarterly → monthly (forward fill)
# ==========================================================

# ----------------------
# FIX 1: colapsar duplicados por ric-period_end
# ----------------------
funda_all = (
    funda_all
    .sort_values(["ric", "period_end"])
    .groupby(["ric", "period_end"], as_index=False)
    .agg({
        "revenue": "mean",
        "industry_sales": "mean",
        "market_share": "mean",
        "gics_sector_name": "first",
        "gics_industry_group_name": "first",
        "gics_industry_name": "first",
        "gics_subindustry_name": "first",
    })
)

print("\n=== QA after collapse ===")
print("Duplicados ric-period_end:", funda_all.duplicated(["ric", "period_end"]).sum())
print("Shape:", funda_all.shape)

# ----------------------
# FIX 2: crear fecha mensual
# ----------------------
funda_all["date"] = funda_all["period_end"].dt.to_period("M").dt.to_timestamp("M")

cols_keep = [
    "ric",
    "date",
    "market_share",
    "industry_sales",
    "revenue",
    "gics_sector_name",
    "gics_industry_group_name",
    "gics_industry_name",
    "gics_subindustry_name",
]

funda_monthly = funda_all[cols_keep].copy()

# ----------------------
# FIX 3: expansión mensual segura (sin resample bug)
# ----------------------
funda_monthly = funda_monthly.sort_values(["ric", "date"])

funda_monthly = (
    funda_monthly
    .groupby("ric", group_keys=False)
    .apply(lambda df: df.set_index("date").resample("M").ffill())
    .reset_index()
)

print("\n=== QA funda_monthly ===")
print("Shape:", funda_monthly.shape)
print("NA market_share:", round(funda_monthly["market_share"].isna().mean(), 4))
print(funda_monthly.head())

# ==========================================================
# 8) Export (MISMO NOMBRE QUE ANTES)
# ==========================================================

# SP500 completo
out_sp500 = OUT.get(
    "fundamentals_with_market_share_sp500",
    RAW_DIR / "fundamentals_with_market_share_sp500.csv"
)

funda_monthly.to_csv(out_sp500, index=False)
logger.info(f"Output -> {out_sp500.as_posix()} | filas={len(funda_monthly):,}")

# muestra (focus)
funda_monthly["ric_base"] = funda_monthly["ric"].astype(str).str.split(".").str[0]

funda_focus = funda_monthly[funda_monthly["ric_base"].isin(target_bases)].copy()

out_focus = OUT.get(
    "fundamentals_with_market_share_focus",
    RAW_DIR / "fundamentals_with_market_share.csv"
)

funda_focus.to_csv(out_focus, index=False)
logger.info(f"Output -> {out_focus.as_posix()} | filas={len(funda_focus):,}")

2026-03-19 18:35:59 | INFO | Muestra (bases RIC): 32
2026-03-19 18:35:59,066 P[23992] [MainThread 21972] Muestra (bases RIC): 32
2026-03-19 18:35:59 | INFO | Descargando universo SP500 (0#.SPX) ...
2026-03-19 18:35:59,069 P[23992] [MainThread 21972] Descargando universo SP500 (0#.SPX) ...


2026-03-19 18:36:01 | INFO | RICs SP500 detectados: 503
2026-03-19 18:36:01,960 P[23992] [MainThread 21972] RICs SP500 detectados: 503
2026-03-19 18:36:01 | INFO | Descargando metadata (GICS) para SP500 ...
2026-03-19 18:36:01,961 P[23992] [MainThread 21972] Descargando metadata (GICS) para SP500 ...
c:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\venv\Lib\site-packages\pandas\core\dtypes\cast.py:1056: RuntimeWarning: invalid value encountered in cast
  if (arr.astype(int) == arr).all():
c:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\venv\Lib\site-packages\pandas\core\dtypes\cast.py:1080: RuntimeWarning: invalid value encountered in cast
  if (arr.astype(int) == arr).all():
2026-03-19 18:37:13 | INFO | Metadata GICS SP500 | filas=503
2026-03-19 18:37:13,953 P[23992] [MainThread 21972] Metadata GICS SP500 | filas=503
2026-03-19 18:37:13 | INFO | Descargando fundamentals (revenue) para SP500 ...
2026-03-19 18:37:13,954 P[23992] [MainThread 21972] Descargando fundamentals (


=== QA after collapse ===
Duplicados ric-period_end: 0
Shape: (5536, 9)

=== QA funda_monthly ===
Shape: (60976, 9)
NA market_share: 0.0
        date  ric  market_share  industry_sales       revenue  \
0 2014-10-31  A.N      0.333333   12144000000.0  4048000000.0   
1 2014-11-30  A.N      0.333333   12144000000.0  4048000000.0   
2 2014-12-31  A.N      0.333333   12144000000.0  4048000000.0   
3 2015-01-31  A.N      0.333333   12144000000.0  4048000000.0   
4 2015-02-28  A.N      0.333333   12144000000.0  4048000000.0   

  gics_sector_name                        gics_industry_group_name  \
0      Health Care  Pharmaceuticals, Biotechnology & Life Sciences   
1      Health Care  Pharmaceuticals, Biotechnology & Life Sciences   
2      Health Care  Pharmaceuticals, Biotechnology & Life Sciences   
3      Health Care  Pharmaceuticals, Biotechnology & Life Sciences   
4      Health Care  Pharmaceuticals, Biotechnology & Life Sciences   

               gics_industry_name           gics_s

2026-03-19 18:39:42 | INFO | Output -> C:/Users/vidar/OneDrive/Desktop/UDESA/Tesis/Tesis-main/data/raw/fundamentals_with_market_share_sp500.csv | filas=60,976
2026-03-19 18:39:42,671 P[23992] [MainThread 21972] Output -> C:/Users/vidar/OneDrive/Desktop/UDESA/Tesis/Tesis-main/data/raw/fundamentals_with_market_share_sp500.csv | filas=60,976
2026-03-19 18:39:42 | INFO | Output -> C:/Users/vidar/OneDrive/Desktop/UDESA/Tesis/Tesis-main/data/raw/fundamentals_with_market_share.csv | filas=3,932
2026-03-19 18:39:42,809 P[23992] [MainThread 21972] Output -> C:/Users/vidar/OneDrive/Desktop/UDESA/Tesis/Tesis-main/data/raw/fundamentals_with_market_share.csv | filas=3,932


## 15bis) Proxies alternativas de market power / competencia

**Motivación.** La variable original de `market_share` fue cuestionada metodológicamente porque su denominador utilizaba solo las firmas de la muestra, por lo que medía participación relativa dentro de la submuestra y no poder de mercado en sentido más amplio. :contentReference[oaicite:1]{index=1}

**Objetivo de esta sección.** Construir proxies más defendibles usando un universo más amplio de comparables y una clasificación GICS más fina.

### Proxies a construir

1. **`market_share_industry_group_sp500`**  
   Participación en ventas de la firma dentro de su **GICS Industry Group** usando el universo del S&P 500.  
   - Ventaja: simple, estable y comparable con la versión ya implementada.
   - Limitación: el denominador sigue siendo SP500, no el universo total de competidores.

2. **`market_share_industry_sp500`**  
   Participación en ventas dentro de **GICS Industry**.  
   - Ventaja: aproxima mejor el mercado relevante que el sector o industry group.
   - Uso sugerido: candidata principal para la tesis si la cobertura es razonable.

3. **`market_share_subindustry_sp500`**  
   Participación en ventas dentro de **GICS Sub-Industry**.  
   - Ventaja: mayor cercanía al mercado de producto.
   - Limitación: puede haber grupos con pocos comparables.

4. **`hhi_industry_sp500`** y `hhi_subindustry_sp500`  
   Índices de concentración construidos como suma de shares al cuadrado dentro de cada industry / subindustry y período.
   - Interpretación: estructura competitiva del mercado donde opera la firma.
   - Uso sugerido: robustez o complemento de market share.

5. **`relative_size_industry_sp500`**  
   Tamaño relativo de la firma frente al total o mediana de ventas de su industria.
   - Interpretación: escala relativa más que poder de mercado.
   - Uso sugerido: robustez si el market share fino tiene poca cobertura.

### Criterio empírico sugerido

- **Proxy principal:** `market_share_industry_sp500`
- **Robustez 1:** `market_share_subindustry_sp500`
- **Robustez 2:** `hhi_industry_sp500`
- **Robustez 3:** `relative_size_industry_sp500`

### Outputs

Esta sección exporta:

- `data/raw/market_power_proxies_sp500_quarterly.csv`
- `data/raw/market_power_proxies_focus_quarterly.csv`

El primer archivo contiene el universo SP500 con las proxies construidas; el segundo filtra a la muestra de firmas de la tesis para luego mergearlo en el pipeline del panel mensual.


Chequeos de los datos. 

In [10]:
# ======================
# MARKET POWER PROXIES (SP500)
# ======================

import numpy as np
import pandas as pd

# -------------------------------------------------
# 0) Base de trabajo
# -------------------------------------------------
req_cols = [
    "ric", "period_end", "revenue",
    "gics_sector_name", "gics_industry_group_name"
]
missing_req = [c for c in req_cols if c not in funda_all.columns]
if missing_req:
    raise RuntimeError(f"Faltan columnas en funda_all: {missing_req}")

req_cols = [
    "ric", "period_end", "revenue",
    "gics_sector_name", "gics_industry_group_name"
]
missing_req = [c for c in req_cols if c not in funda_all.columns]
if missing_req:
    raise RuntimeError(f"Faltan columnas en funda_all: {missing_req}")

print("\nCobertura columnas GICS finas en funda_all:")
for c in ["gics_industry_name", "gics_subindustry_name"]:
    print(f"{c}: {c in funda_all.columns}")

mp = funda_all.copy()
mp["period_end"] = pd.to_datetime(mp["period_end"], errors="coerce")
mp["revenue"] = pd.to_numeric(mp["revenue"], errors="coerce")
mp = mp.dropna(subset=["ric", "period_end"]).copy()

# -------------------------------------------------
# FIX 1: colapsar duplicados (CRÍTICO)
# -------------------------------------------------
agg_dict = {
    "revenue": "mean",
    "gics_sector_name": "first",
    "gics_industry_group_name": "first",
}

if "gics_industry_name" in mp.columns:
    agg_dict["gics_industry_name"] = "first"

if "gics_subindustry_name" in mp.columns:
    agg_dict["gics_subindustry_name"] = "first"

mp = (
    mp
    .sort_values(["ric", "period_end"])
    .groupby(["ric", "period_end"], as_index=False)
    .agg(agg_dict)
)

print("\n=== QA mp after collapse ===")
print("Duplicados:", mp.duplicated(["ric", "period_end"]).sum())

# Limpieza robusta
for c in [
    "gics_sector_name",
    "gics_industry_group_name",
    "gics_industry_name",
    "gics_subindustry_name"
]:
    if c in mp.columns:
        mp[c] = mp[c].astype("string").str.strip()

mp.loc[mp["revenue"] <= 0, "revenue"] = np.nan

# -------------------------------------------------
# 1) Helper robusto
# -------------------------------------------------
def build_share_hhi(df, group_col, suffix):

    tmp = df[["ric", "period_end", "revenue", group_col]].copy()
    tmp = tmp.dropna(subset=["period_end", group_col])

    tmp_valid = tmp.dropna(subset=["revenue"]).copy()

    grp = (
        tmp_valid
        .groupby([group_col, "period_end"], as_index=False)
        .agg(
            group_sales=("revenue", "sum"),
            group_median_sales=("revenue", "median"),
            n_firms=("ric", "nunique")
        )
    )

    grp = grp[grp["group_sales"] > 0]

    out = tmp.merge(grp, on=[group_col, "period_end"], how="left")

    out[f"market_share_{suffix}"] = out["revenue"] / out["group_sales"]
    out[f"relative_size_{suffix}"] = out["revenue"] / out["group_median_sales"]

    out[f"rank_revenue_{suffix}"] = (
        out.groupby([group_col, "period_end"])["revenue"]
        .rank(method="dense", ascending=False)
    )

    hhi = (
        out.dropna(subset=[f"market_share_{suffix}"])
        .assign(share_sq=lambda x: x[f"market_share_{suffix}"] ** 2)
        .groupby([group_col, "period_end"], as_index=False)["share_sq"]
        .sum()
        .rename(columns={"share_sq": f"hhi_{suffix}"})
    )

    out = out.merge(hhi, on=[group_col, "period_end"], how="left")

    out = out.rename(columns={
        "group_sales": f"group_sales_{suffix}",
        "group_median_sales": f"group_median_sales_{suffix}",
        "n_firms": f"n_firms_{suffix}"
    })

    keep = [
        "ric", "period_end",
        f"group_sales_{suffix}",
        f"group_median_sales_{suffix}",
        f"n_firms_{suffix}",
        f"market_share_{suffix}",
        f"relative_size_{suffix}",
        f"rank_revenue_{suffix}",
        f"hhi_{suffix}",
    ]

    return out[keep].drop_duplicates(subset=["ric", "period_end"])


# -------------------------------------------------
# 2) Construcción proxies (ROBUSTO)
# -------------------------------------------------
logger.info("Construyendo proxies market power ...")

proxies = []

# siempre existe
proxies.append(
    build_share_hhi(mp, "gics_industry_group_name", "industry_group_sp500")
)

# opcionales
if "gics_industry_name" in mp.columns:
    proxies.append(
        build_share_hhi(mp, "gics_industry_name", "industry_sp500")
    )

if "gics_subindustry_name" in mp.columns:
    proxies.append(
        build_share_hhi(mp, "gics_subindustry_name", "subindustry_sp500")
    )

# -------------------------------------------------
# 3) Merge horizontal
# -------------------------------------------------
base_cols = [
    c for c in [
        "ric", "period_end", "revenue",
        "gics_sector_name",
        "gics_industry_group_name",
        "gics_industry_name",
        "gics_subindustry_name"
    ] if c in mp.columns
]

market_power = mp[base_cols].copy()

for piece in proxies:
    market_power = market_power.merge(
        piece,
        on=["ric", "period_end"],
        how="left"
    )

# -------------------------------------------------
# 4) Variables extra
# -------------------------------------------------
market_power["log_revenue"] = np.log(market_power["revenue"])

if "n_firms_subindustry_sp500" in market_power.columns:
    market_power["small_subindustry_group"] = (
        market_power["n_firms_subindustry_sp500"].fillna(0) < 4
    ).astype(int)

if "n_firms_industry_sp500" in market_power.columns:
    market_power["small_industry_group"] = (
        market_power["n_firms_industry_sp500"].fillna(0) < 4
    ).astype(int)

market_power["ric_base"] = market_power["ric"].astype(str).str.split(".").str[0]

# -------------------------------------------------
# 5) Quarterly → Monthly
# -------------------------------------------------
market_power = market_power.sort_values(["ric", "period_end"])

market_power["date"] = market_power["period_end"].dt.to_period("M").dt.to_timestamp("M")

market_power = (
    market_power
    .groupby("ric", group_keys=False)
    .apply(lambda df: df.set_index("date").resample("M").ffill())
    .reset_index()
)

print("\n=== QA market_power monthly ===")
print("NA IG:", market_power["market_share_industry_group_sp500"].isna().mean())

if "market_share_industry_sp500" in market_power.columns:
    print("NA IND:", market_power["market_share_industry_sp500"].isna().mean())

if "market_share_subindustry_sp500" in market_power.columns:
    print("NA SUB:", market_power["market_share_subindustry_sp500"].isna().mean())

# -------------------------------------------------
# 6) Export
# -------------------------------------------------
out_all = OUT.get(
    "market_power_proxies_sp500_q",
    RAW_DIR / "market_power_proxies_sp500_quarterly.csv"
)
market_power.to_csv(out_all, index=False)

market_power_focus = market_power[market_power["ric_base"].isin(target_bases)].copy()

out_focus = OUT.get(
    "market_power_proxies_focus_q",
    RAW_DIR / "market_power_proxies_focus_quarterly.csv"
)
market_power_focus.to_csv(out_focus, index=False)

# -------------------------------------------------
# 7) Coverage
# -------------------------------------------------
summary_cols = [c for c in [
    "market_share_industry_group_sp500",
    "market_share_industry_sp500",
    "market_share_subindustry_sp500",
    "hhi_industry_group_sp500",
    "hhi_industry_sp500",
    "hhi_subindustry_sp500",
] if c in market_power_focus.columns]

coverage = (
    market_power_focus[summary_cols]
    .notna()
    .mean()
    .sort_values(ascending=False)
)

print("\n=== COVERAGE ===")
print(coverage)

2026-03-19 18:43:28 | INFO | Construyendo proxies market power ...
2026-03-19 18:43:28,269 P[23992] [MainThread 21972] Construyendo proxies market power ...



Cobertura columnas GICS finas en funda_all:
gics_industry_name: True
gics_subindustry_name: True

=== QA mp after collapse ===
Duplicados: 0

=== QA market_power monthly ===
NA IG: 0.0
NA IND: 0.0
NA SUB: 0.0

=== COVERAGE ===
market_share_industry_group_sp500    1.0
market_share_industry_sp500          1.0
market_share_subindustry_sp500       1.0
hhi_industry_group_sp500             1.0
hhi_industry_sp500                   1.0
hhi_subindustry_sp500                1.0
dtype: float64


In [11]:
# ==========================================================
# 1) Distribución del número de firmas por grupo (ROBUSTO)
# ==========================================================
cols_n_firms = [
    "n_firms_industry_group_sp500",
    "n_firms_industry_sp500",
    "n_firms_subindustry_sp500"
]

for c in cols_n_firms:
    if c in market_power_focus.columns:
        print("\n", c)
        print(market_power_focus[c].describe())
        print(market_power_focus[c].value_counts().sort_index().head(10))
    else:
        print(f"\n{c} -> NO EXISTE")

# ==========================================================
# 2) Casos donde HHI = 1 (ROBUSTO)
# ==========================================================
cols_hhi = [
    "hhi_industry_group_sp500",
    "hhi_industry_sp500",
    "hhi_subindustry_sp500"
]

for c in cols_hhi:
    if c not in market_power.columns:
        print(f"\n{c} -> NO EXISTE")
        continue

    print(f"\nCasos con {c} = 1")

    cols_show = ["ric", "period_end", c]

    # agregar columnas si existen
    extra_cols = [
        "gics_industry_group_name",
        "gics_industry_name",
        "gics_subindustry_name",
        "n_firms_industry_group_sp500",
        "n_firms_industry_sp500",
        "n_firms_subindustry_sp500"
    ]

    cols_show += [col for col in extra_cols if col in market_power.columns]

    print(market_power.loc[market_power[c] == 1, cols_show].head(20))
    print("Total:", (market_power[c] == 1).sum())

# ==========================================================
# 3) Verificar duplicados ric-period_end
# ==========================================================
dup_check = (
    mp.groupby(["ric", "period_end"])
      .size()
      .reset_index(name="n")
      .query("n > 1")
)

print("\nDuplicados ric-period_end:", len(dup_check))
print(dup_check.head(20))


 n_firms_industry_group_sp500
count    3932.000000
mean       18.057223
std        13.019740
min         1.000000
25%        10.000000
50%        15.000000
75%        29.000000
max        44.000000
Name: n_firms_industry_group_sp500, dtype: float64
n_firms_industry_group_sp500
1     628
2     207
3     133
4      12
10     97
11     12
12    302
13    496
14     73
15     12
Name: count, dtype: int64

 n_firms_industry_sp500
count    3932.000000
mean        9.395219
std         7.225654
min         1.000000
25%         2.000000
50%         8.000000
75%        16.000000
max        23.000000
Name: n_firms_industry_sp500, dtype: float64
n_firms_industry_sp500
1     822
2     388
4     351
5      48
6     194
7      96
8      97
9      85
10     36
11     24
Name: count, dtype: int64

 n_firms_subindustry_sp500
count    3932.000000
mean        5.317650
std         4.453504
min         1.000000
25%         2.000000
50%         4.000000
75%         8.000000
max        17.000000
Name: n_firm

## 16) Controles macroeconómicos agregados — descarga y construcción mensual

**Objetivo:** incorporar un set parsimonioso de controles macro para los modelos sin efectos fijos de tiempo, de modo de evaluar si los coeficientes de los shocks agregados de mercado y crédito son robustos a otras condiciones macrofinancieras comunes.

### Criterio de selección

Se priorizan controles con interpretación económica clara y bajo solapamiento con los factores ya incluidos en la tesis.

**Set mínimo sugerido para regresiones:**
- `policy_rate`: tasa de política monetaria de la Fed (o proxy de corto plazo si la serie directa no está disponible).
- `term_spread_10y_2y`: pendiente de la curva del Tesoro.
- `indpro_yoy`: actividad económica agregada.

**Set ampliado de robustez:**
- `cpi_yoy`: inflación interanual.
- `vix_eom`: incertidumbre / aversión al riesgo agregada.
- `move_eom`: volatilidad implícita del mercado de Treasuries.

### Notas metodológicas

- La pendiente del Tesoro se construye a partir de la curva UST ya descargada en la etapa anterior.
- Para la tasa de política se intenta primero una serie macro directa; si no hay cobertura por permisos o por RIC, se construye una **proxy** con la tasa UST de 3 meses.
- Las series diarias se agregan a frecuencia mensual usando **último dato disponible de cada mes**.
- Las series macro mensuales se guardan ya listas para merge con el panel firma–mes.

### Outputs

- `data/raw/macro_controls_daily.parquet`
- `data/clean/macro_controls_monthly.parquet`
- `data/clean/macro_controls_sources.csv`

### Observación importante

Los RICs de algunas series macroeconómicas en Refinitiv pueden variar según permisos, Workspace o convención del monitor económico. Por eso la celda prueba varios candidatos por variable y registra cuál funcionó. Si una serie no baja, normalmente alcanza con reemplazar la lista de candidatos de esa variable por el RIC correcto encontrado en Workspace.


In [12]:
# ======================
# CONTROLES MACRO -> descarga + construcción mensual
# ======================

# Requiere:
# - START_DATE, END_DATE
# - RAW_DIR, CLEAN_DIR, OUT
# - logger, sleep_request
# - get_ts_safe(...) de la celda de UST
# - ust_yield_curve_daily.parquet ya generado

from typing import Optional

# -------------------------------------------------
# 1) Configuración
# -------------------------------------------------

# Variables seleccionadas:
# - baseline: policy_rate, term_spread_10y_2y, indpro_yoy
# - robustez: cpi_yoy, vix_eom, move_eom
#
# Nota:
# - TED spread NO se incluye acá porque su construcción homogénea hasta 2025
#   es menos limpia por el quiebre de LIBOR. Si después querés, lo armamos
#   como robustness separado.

# Candidatos de RIC por variable.
# IMPORTANTE:
# Los RICs macro pueden variar según permisos / Workspace.
# Si una serie no baja, buscá el RIC en Workspace y reemplazá esta lista.
MACRO_RIC_CANDIDATES = {
    # incertidumbre / robustez
    "vix": [
        ".VIX",
        "VIX"
    ],
    "move": [
        ".MOVE",
        "MOVE"
    ],

    # política monetaria (si no funciona ninguno, se usa proxy UST 3M)
    "policy_rate": [
        "USFFTARGET=",      # candidato 1
        "USFEDFUNDS=",      # candidato 2
        "FEDFUNDS"          # candidato 3
    ],

    # inflación interanual (mensual)
    "cpi_yoy": [
        "USCPIY=ECI",       # candidato 1
        "aUSCPIY",          # candidato 2
        "USCPNY=ECI"        # candidato 3
    ],

    # actividad real (mensual)
    "indpro_yoy": [
        "USIPY=ECI",        # candidato 1
        "aUSIPY",           # candidato 2
        "USIP=ECI"          # candidato 3 (si devuelve índice nivel, se transforma)
    ],
}

# frecuencia deseada por variable descargada directamente
MACRO_FREQ = {
    "vix": "daily",
    "move": "daily",
    "policy_rate": "monthly",
    "cpi_yoy": "monthly",
    "indpro_yoy": "monthly",
}

# -------------------------------------------------
# 2) Helpers
# -------------------------------------------------

def normalize_ts_output(ts: pd.DataFrame, var_name: str, ric_used: str) -> pd.DataFrame:
    """
    Normaliza la salida de ek.get_timeseries a columnas:
    date, value, variable, ric
    """
    if ts is None or ts.empty:
        return pd.DataFrame(columns=["date", "value", "variable", "ric"])

    df = ts.reset_index().copy()

    # detectar columna fecha
    date_col = None
    for c in df.columns:
        if str(c).lower() in {"date", "index"}:
            date_col = c
            break
    if date_col is None:
        date_col = df.columns[0]

    # detectar columna valor
    preferred = ["CLOSE", "VALUE", "OPEN", "HIGH", "LOW"]
    value_col = None
    for c in preferred:
        if c in df.columns:
            value_col = c
            break

    if value_col is None:
        # primer numérico disponible distinto de date
        candidates = [c for c in df.columns if c != date_col]
        if not candidates:
            return pd.DataFrame(columns=["date", "value", "variable", "ric"])
        value_col = candidates[0]

    out = df[[date_col, value_col]].rename(columns={date_col: "date", value_col: "value"}).copy()
    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    if getattr(out["date"].dt, "tz", None) is not None:
        out["date"] = out["date"].dt.tz_localize(None)

    out["value"] = pd.to_numeric(out["value"], errors="coerce")
    out["variable"] = var_name
    out["ric"] = ric_used

    out = out.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)
    return out


def fetch_first_valid_series(
    var_name: str,
    candidates: list[str],
    start_date: str,
    end_date: str,
    interval: str,
    max_attempts: int = 3,
) -> tuple[pd.DataFrame, Optional[str]]:
    """
    Prueba varios RICs y devuelve la primera serie válida.
    """
    for ric in candidates:
        logger.info(f"[macro] {var_name} -> probando {ric} ({interval})")
        try:
            ts = ek.get_timeseries(
                ric,
                fields=["CLOSE"],
                start_date=start_date,
                end_date=end_date,
                interval=interval,
            )
            df = normalize_ts_output(ts, var_name=var_name, ric_used=ric)
            if not df.empty and df["value"].notna().sum() > 0:
                logger.info(f"[macro] OK {var_name} con {ric} | filas={len(df):,}")
                return df, ric

        except Exception as e:
            msg = str(e)
            if "429" in msg or "Too many requests" in msg:
                wait_sec = 4
                logger.warning(f"[macro] 429 para {var_name}/{ric} | espero {wait_sec}s")
                time.sleep(wait_sec)
                continue
            logger.warning(f"[macro] error {var_name}/{ric}: {type(e).__name__}: {e}")

        sleep_request(0.4)

    logger.warning(f"[macro] sin cobertura para {var_name}")
    return pd.DataFrame(columns=["date", "value", "variable", "ric"]), None


def monthly_last(df: pd.DataFrame, date_col: str, value_col: str, out_name: str) -> pd.DataFrame:
    """
    Último dato disponible de cada mes.
    """
    x = df[[date_col, value_col]].copy()
    x[date_col] = pd.to_datetime(x[date_col], errors="coerce")
    x = x.dropna(subset=[date_col]).sort_values(date_col)
    x["date"] = x[date_col].dt.to_period("M").dt.to_timestamp("M")
    x = x.groupby("date", as_index=False)[value_col].last()
    x = x.rename(columns={value_col: out_name})
    return x


def monthly_mean(df: pd.DataFrame, date_col: str, value_col: str, out_name: str) -> pd.DataFrame:
    """
    Promedio mensual.
    """
    x = df[[date_col, value_col]].copy()
    x[date_col] = pd.to_datetime(x[date_col], errors="coerce")
    x = x.dropna(subset=[date_col]).sort_values(date_col)
    x["date"] = x[date_col].dt.to_period("M").dt.to_timestamp("M")
    x = x.groupby("date", as_index=False)[value_col].mean()
    x = x.rename(columns={value_col: out_name})
    return x


# -------------------------------------------------
# 3) Descargar series macro directas
# -------------------------------------------------

logger.info("Descargando controles macro directos...")

macro_long = []
macro_sources = []

for var_name, ric_list in MACRO_RIC_CANDIDATES.items():
    interval = MACRO_FREQ[var_name]
    df_var, ric_used = fetch_first_valid_series(
        var_name=var_name,
        candidates=ric_list,
        start_date=START_DATE,
        end_date=END_DATE,
        interval=interval,
        max_attempts=RETRY_MAX,
    )

    if not df_var.empty:
        macro_long.append(df_var)

    macro_sources.append({
        "variable": var_name,
        "ric_used": ric_used,
        "download_status": "ok" if ric_used is not None else "missing",
        "frequency_requested": interval,
    })

    sleep_request(0.5)

macro_sources = pd.DataFrame(macro_sources)

if macro_long:
    macro_raw = pd.concat(macro_long, ignore_index=True)
else:
    macro_raw = pd.DataFrame(columns=["date", "value", "variable", "ric"])

# Export raw
raw_macro_parquet = OUT.get("macro_controls_daily", RAW_DIR / "macro_controls_daily.parquet")
raw_macro_csv = raw_macro_parquet.with_suffix(".csv")

safe_write_parquet(macro_raw, raw_macro_parquet, index=False)
macro_raw.to_csv(raw_macro_csv, index=False)
logger.info(f"Output raw macro -> {raw_macro_csv.as_posix()}")

# -------------------------------------------------
# 4) Construir UST monthly + policy proxy + term spread
# -------------------------------------------------

ust_path = OUT.get("ust_yield_curve_daily", RAW_DIR / "ust_yield_curve_daily.parquet")
if not ust_path.exists():
    raise RuntimeError(
        "No existe ust_yield_curve_daily.parquet. "
        "Primero corré la celda de curva UST."
    )

ust = pd.read_parquet(ust_path).copy()
ust["date"] = pd.to_datetime(ust["date"], errors="coerce")

# último dato disponible por mes y tenor
ust_m = (
    ust.dropna(subset=["date", "tenor", "yield"])
       .sort_values(["tenor", "date"])
       .assign(date=lambda x: x["date"].dt.to_period("M").dt.to_timestamp("M"))
       .groupby(["date", "tenor"], as_index=False)["yield"].last()
       .pivot(index="date", columns="tenor", values="yield")
       .reset_index()
)

# normalizar nombres
ust_m = ust_m.rename(columns={
    "US3M": "ust_3m",
    "US2Y": "ust_2y",
    "US10Y": "ust_10y",
})

# spreads risk-free
if {"ust_10y", "ust_2y"}.issubset(ust_m.columns):
    ust_m["term_spread_10y_2y"] = ust_m["ust_10y"] - ust_m["ust_2y"]

if {"ust_10y", "ust_3m"}.issubset(ust_m.columns):
    ust_m["term_spread_10y_3m"] = ust_m["ust_10y"] - ust_m["ust_3m"]

# -------------------------------------------------
# 5) Pasar series directas a monthly
# -------------------------------------------------

monthly_parts = [ust_m[["date"] + [c for c in ust_m.columns if c != "date"]].copy()]

if not macro_raw.empty:
    # VIX -> último del mes
    if (macro_raw["variable"] == "vix").any():
        vix_m = monthly_last(
            macro_raw.loc[macro_raw["variable"] == "vix"],
            date_col="date",
            value_col="value",
            out_name="vix_eom",
        )
        monthly_parts.append(vix_m)

    # MOVE -> último del mes
    if (macro_raw["variable"] == "move").any():
        move_m = monthly_last(
            macro_raw.loc[macro_raw["variable"] == "move"],
            date_col="date",
            value_col="value",
            out_name="move_eom",
        )
        monthly_parts.append(move_m)

    # policy rate directo (mensual)
    if (macro_raw["variable"] == "policy_rate").any():
        pol_m = monthly_last(
            macro_raw.loc[macro_raw["variable"] == "policy_rate"],
            date_col="date",
            value_col="value",
            out_name="policy_rate",
        )
        monthly_parts.append(pol_m)

    # CPI YoY (mensual)
    if (macro_raw["variable"] == "cpi_yoy").any():
        cpi_m = monthly_last(
            macro_raw.loc[macro_raw["variable"] == "cpi_yoy"],
            date_col="date",
            value_col="value",
            out_name="cpi_yoy",
        )
        monthly_parts.append(cpi_m)

    # INDPRO YoY (mensual)
    if (macro_raw["variable"] == "indpro_yoy").any():
        ip_m = monthly_last(
            macro_raw.loc[macro_raw["variable"] == "indpro_yoy"],
            date_col="date",
            value_col="value",
            out_name="indpro_yoy",
        )
        monthly_parts.append(ip_m)

# merge outer sobre date
macro_monthly = None
for part in monthly_parts:
    if macro_monthly is None:
        macro_monthly = part.copy()
    else:
        macro_monthly = macro_monthly.merge(part, on="date", how="outer")

macro_monthly = macro_monthly.sort_values("date").reset_index(drop=True)

# -------------------------------------------------
# 6) Fallback para tasa de política
# -------------------------------------------------

# Si no se pudo bajar una serie directa de policy_rate, usar proxy UST 3M.
if "policy_rate" not in macro_monthly.columns or macro_monthly["policy_rate"].notna().sum() == 0:
    if "ust_3m" in macro_monthly.columns:
        macro_monthly["policy_rate"] = macro_monthly["ust_3m"]
        macro_sources = pd.concat([
            macro_sources,
            pd.DataFrame([{
                "variable": "policy_rate_fallback",
                "ric_used": "UST_3M_proxy_from_ust_yield_curve_daily",
                "download_status": "constructed_proxy",
                "frequency_requested": "monthly"
            }])
        ], ignore_index=True)
        logger.warning("policy_rate directo no disponible -> se usa proxy 'ust_3m'.")
    else:
        logger.warning("No hay policy_rate directo ni proxy ust_3m disponible.")

# -------------------------------------------------
# 7) Si INDPRO vino en nivel, transformar a YoY
# -------------------------------------------------

# Si por permisos el candidato de indpro devuelve nivel y no tasa YoY,
# transformarlo automáticamente.
if "indpro_yoy" in macro_monthly.columns:
    # heurística simple: si los valores parecen índice nivel (> 50 de forma persistente),
    # convertir a variación interanual.
    s = pd.to_numeric(macro_monthly["indpro_yoy"], errors="coerce")
    if s.notna().sum() >= 24 and s.median() > 20:
        macro_monthly = macro_monthly.sort_values("date").reset_index(drop=True)
        macro_monthly["indpro_yoy"] = 100 * (macro_monthly["indpro_yoy"] / macro_monthly["indpro_yoy"].shift(12) - 1)
        logger.info("indpro_yoy fue transformada desde índice nivel a variación interanual (%).")

# -------------------------------------------------
# 8) Rango final y columnas sugeridas
# -------------------------------------------------

macro_monthly = macro_monthly[
    (macro_monthly["date"] >= pd.to_datetime(START_DATE))
    & (macro_monthly["date"] <= pd.to_datetime(END_DATE))
].copy()

preferred_order = [
    "date",
    # baseline
    "policy_rate",
    "term_spread_10y_2y",
    "indpro_yoy",
    # robustness
    "cpi_yoy",
    "vix_eom",
    "move_eom",
    # auxiliares / QA
    "ust_3m",
    "ust_2y",
    "ust_10y",
    "term_spread_10y_3m",
]

existing_cols = [c for c in preferred_order if c in macro_monthly.columns]
other_cols = [c for c in macro_monthly.columns if c not in existing_cols]
macro_monthly = macro_monthly[existing_cols + other_cols]

# -------------------------------------------------
# 9) Export
# -------------------------------------------------

out_parquet = OUT.get("macro_controls_monthly", CLEAN_DIR / "macro_controls_monthly.parquet")
out_csv = out_parquet.with_suffix(".csv")
src_csv = CLEAN_DIR / "macro_controls_sources.csv"

safe_write_parquet(macro_monthly, out_parquet, index=False)
macro_monthly.to_csv(out_csv, index=False)
macro_sources.to_csv(src_csv, index=False)

logger.info(f"Output monthly macro -> {out_csv.as_posix()}")
logger.info(f"Output sources      -> {src_csv.as_posix()}")

# -------------------------------------------------
# 10) Resumen
# -------------------------------------------------

print("\n=== RESUMEN CONTROLES MACRO ===")
print("Columnas disponibles:")
print(macro_monthly.columns.tolist())

print("\nCobertura no nula por variable:")
print(macro_monthly.notna().sum().sort_values(ascending=False))

print("\nPrimeras filas:")
display(macro_monthly.head())

print("\nFuentes / RICs usados:")
display(macro_sources)


2026-03-19 18:43:47 | INFO | Descargando controles macro directos...
2026-03-19 18:43:47,108 P[23992] [MainThread 21972] Descargando controles macro directos...
2026-03-19 18:43:47 | INFO | [macro] vix -> probando .VIX (daily)
2026-03-19 18:43:47,111 P[23992] [MainThread 21972] [macro] vix -> probando .VIX (daily)
2026-03-19 18:43:47,634 P[23992] [MainThread 21972] Error with .VIX: The user does not have permission for the requested data
2026-03-19 18:43:47,635 P[23992] [MainThread 21972] .VIX: The user does not have permission for the requested data | 
2026-03-19 18:43:47 | WARNING | [macro] error vix/.VIX: EikonError: Error code -1 | .VIX: The user does not have permission for the requested data | 
2026-03-19 18:43:47,636 P[23992] [MainThread 21972] [macro] error vix/.VIX: EikonError: Error code -1 | .VIX: The user does not have permission for the requested data | 
2026-03-19 18:43:48 | INFO | [macro] vix -> probando VIX (daily)
2026-03-19 18:43:48,039 P[23992] [MainThread 21972] [ma


=== RESUMEN CONTROLES MACRO ===
Columnas disponibles:
['date', 'policy_rate', 'term_spread_10y_2y', 'move_eom', 'ust_3m', 'ust_2y', 'ust_10y', 'term_spread_10y_3m', 'US1M', 'US1Y', 'US30Y', 'US3Y', 'US5Y', 'US6M', 'US7Y']

Cobertura no nula por variable:
date                  132
policy_rate           132
term_spread_10y_2y    132
move_eom              132
ust_3m                132
ust_2y                132
ust_10y               132
term_spread_10y_3m    132
US1M                  132
US1Y                  132
US30Y                 132
US3Y                  132
US5Y                  132
US6M                  132
US7Y                  132
dtype: int64

Primeras filas:


,date,policy_rate,term_spread_10y_2y,move_eom,ust_3m,ust_2y,ust_10y,term_spread_10y_3m,US1M,US1Y,US30Y,US3Y,US5Y,US6M,US7Y
0,2015-01-31,0.25,1.208,86.6,0.013,0.472,1.68,1.667,0.005,0.147,2.258,0.768,1.19,0.051,1.496
1,2015-02-28,0.25,1.376,91.4,0.015,0.626,2.002,1.987,0.018,0.193,2.6,1.0,1.504,0.074,1.823
2,2015-03-31,0.25,1.375,86.1,0.036,0.559,1.934,1.898,0.018,0.229,2.543,0.882,1.375,0.137,1.714
3,2015-04-30,0.25,1.465,75.3,0.005,0.579,2.044,2.039,-0.003,0.232,2.752,0.914,1.44,0.038,1.804
4,2015-05-31,0.25,1.49,82.7,0.003,0.605,2.095,2.092,0.008,0.25,2.847,0.922,1.469,0.064,1.844



Fuentes / RICs usados:


,variable,ric_used,download_status,frequency_requested
0,vix,None,missing,daily
1,move,.MOVE,ok,daily
2,policy_rate,USFFTARGET=,ok,monthly
3,cpi_yoy,None,missing,monthly
4,indpro_yoy,None,missing,monthly
